In [1]:
%pip install -q onnx onnxruntime

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, json, math, time, zlib, random, copy, inspect, warnings, hashlib
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from scipy import ndimage
from scipy.stats import rankdata, wilcoxon, spearmanr, binomtest
import matplotlib.pyplot as plt


MVTEC_ROOT = None
PIPELINE_VERSION = 5
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

MVTEC_CATEGORIES = ["bottle", "cable", "capsule", "carpet", "grid", "hazelnut", "leather", "metal_nut",
                    "pill", "screw", "tile", "toothbrush", "transistor", "wood", "zipper"]

FAMILY_GROUP = {"gamma": "photometric", "color_temperature": "photometric",
                "gaussian_noise": "sensor", "motion_blur": "sensor", "jpeg": "sensor",
                "rotation": "geometric", "perspective": "geometric"}

SEVERITY_TABLE = {"gamma": [1.4, 1.8, 2.4],
                  "color_temperature": [0.10, 0.20, 0.35],
                  "gaussian_noise": [8.0, 16.0, 28.0],
                  "motion_blur": [5, 9, 15],
                  "jpeg": [40, 20, 10],
                  "rotation": [5.0, 10.0, 20.0],
                  "perspective": [0.04, 0.08, 0.14]}

DETECTOR_SPECS = {"patchcore_wrn": dict(kind="patchcore", view="wrn", aug=False),
                  "patchcore_wrn_aug": dict(kind="patchcore", view="wrn", aug=True),
                  "patchcore_wrn_photonorm": dict(kind="patchcore", view="wrn_photo", aug=False),
                  "patchcore_dino": dict(kind="patchcore", view="dino", aug=False),
                  "patchcore_dino_aug": dict(kind="patchcore", view="dino", aug=True),
                  "padim_wrn": dict(kind="padim", view="wrn", aug=False),
                  "padim_wrn_aug": dict(kind="padim", view="wrn", aug=True)}


def base_config():
    working = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
    return dict(
        mvtec_root=MVTEC_ROOT, search_roots=["/kaggle/input", "/tmp/mvtec_extracted", "./data"],
        extract_dir="/tmp/mvtec_extracted", previous_output_roots=["/kaggle/input"],
        output_dir=os.path.join(working, "robust_ad_v4"),
        categories=list(MVTEC_CATEGORIES),
        image_size=256, crop_size=224, pixel_eval_size=112, blur_sigma=4.0,
        calibration_fraction=0.2, max_train_images=None, max_test_images=None,
        coreset_fraction=0.1, coreset_cap=16000, coreset_projection=128,
        padim_dims=100, padim_eps=0.01, valid_margins=[16, 32, 48],
        dino_name="vit_small_patch14_dinov2.lvd142m", dino_layers=(7, 11),
        pretrained=True, alpha=0.1, batch_size=32, seed=0,
        seen_families=["gamma", "gaussian_noise", "rotation"],
        heldout_families=["color_temperature", "motion_blur", "jpeg", "perspective"],
        severities=[1, 2, 3],
        geometry_semantic=["transistor"],
        detectors=list(DETECTOR_SPECS),
        gallery_detector="patchcore_wrn", gallery_k=6,
        deploy_category="bottle", verify_cpp=True, bootstrap=5000, overwrite=False,
    )


CONFIG = base_config()


def stable_seed(*parts):
    return zlib.crc32("|".join(str(p) for p in parts).encode()) & 0x7FFFFFFF


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed % (2 ** 32))
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


def log(message):
    stamp = time.strftime("%H:%M:%S")
    line = f"[{stamp}] {message}"
    print(line, flush=True)
    path = Path(CONFIG["output_dir"])
    path.mkdir(parents=True, exist_ok=True)
    with open(path / "run_log.txt", "a") as handle:
        handle.write(line + "\n")


print(f"Running on {DEVICE} with torch {torch.__version__} and torchvision {torchvision.__version__}")

Running on cuda:0 with torch 2.11.0+cu128 and torchvision 0.26.0+cu128


In [3]:
# data loading

IMAGE_EXT = (".png", ".jpg", ".jpeg", ".bmp")
_ROOT_CACHE = {}


ARCHIVE_EXT = (".tar.xz", ".tar.gz", ".tgz", ".tar.bz2", ".tar", ".zip")


def is_mvtec_dir(path):
    return any(os.path.isdir(os.path.join(path, c, "train", "good")) for c in MVTEC_CATEGORIES)


def _scan(base, max_depth=8):
    found, archives, seen = None, [], set()
    stack = [(base, 0)]
    while stack:
        path, depth = stack.pop()
        real = os.path.realpath(path)
        if real in seen:
            continue
        seen.add(real)
        if is_mvtec_dir(path):
            return path, archives
        try:
            entries = sorted(os.scandir(path), key=lambda e: e.name)
        except (PermissionError, FileNotFoundError, NotADirectoryError):
            continue
        for entry in entries:
            name = entry.name
            if name.startswith("."):
                continue
            if entry.is_file(follow_symlinks=True) and name.lower().endswith(ARCHIVE_EXT):
                archives.append(entry.path)
            elif entry.is_dir(follow_symlinks=True) and depth < max_depth and \
                    name not in ("train", "test", "ground_truth") and name not in MVTEC_CATEGORIES:
                stack.append((entry.path, depth + 1))
    return found, archives


def safe_extract(archive, target):
    import tarfile, zipfile
    root = os.path.realpath(target)
    if archive.lower().endswith(".zip"):
        with zipfile.ZipFile(archive) as handle:
            names = handle.namelist()
            bad = [n for n in names if not os.path.realpath(os.path.join(root, n)).startswith(root + os.sep)]
            if bad:
                raise ValueError(f"{archive} has entries outside the target folder, e.g. {bad[0]}")
            handle.extractall(root)
        return
    with tarfile.open(archive, "r:*") as handle:
        if "filter" in inspect.signature(handle.extractall).parameters:
            handle.extractall(root, filter="data")
            return
        for member in handle.getmembers():
            path = os.path.realpath(os.path.join(root, member.name))
            if not path.startswith(root + os.sep) or member.issym() or member.islnk():
                raise ValueError(f"{archive} has an unsafe entry: {member.name}")
        handle.extractall(root)


def _extract(archives, target):
    os.makedirs(target, exist_ok=True)
    for archive in archives:
        marker = os.path.join(target, "." + os.path.basename(archive) + ".done")
        if os.path.exists(marker):
            continue
        print(f"Extracting {os.path.basename(archive)} to {target}. This takes a few minutes the first time.", flush=True)
        safe_extract(archive, target)
        open(marker, "w").close()


def _tree(base, depth=4, limit=60):
    lines = []
    def walk(path, level):
        if level > depth or len(lines) >= limit:
            return
        try:
            entries = sorted(os.scandir(path), key=lambda e: e.name)
        except Exception:
            return
        for entry in entries[:12]:
            if len(lines) >= limit:
                return
            kind = "/" if entry.is_dir(follow_symlinks=True) else ""
            lines.append("  " * level + entry.name + kind)
            if kind:
                walk(entry.path, level + 1)
    walk(base, 0)
    return "\n".join(lines) if lines else "(empty)"


def find_mvtec_root():
    hint = CONFIG["mvtec_root"]
    if hint:
        if is_mvtec_dir(hint):
            return hint
        raise FileNotFoundError(f"mvtec_root {hint} has no <category>/train/good folder. Contents:\n{_tree(hint, 2)}")
    if "root" in _ROOT_CACHE:
        return _ROOT_CACHE["root"]
    archives = []
    for base in CONFIG["search_roots"]:
        if os.path.isdir(base):
            root, found_archives = _scan(base)
            if root:
                _ROOT_CACHE["root"] = root
                print(f"MVTec AD found at {root}", flush=True)
                return root
            archives += found_archives
    if archives:
        _extract(sorted(set(archives)), CONFIG["extract_dir"])
        root, _ = _scan(CONFIG["extract_dir"])
        if root:
            _ROOT_CACHE["root"] = root
            print(f"MVTec AD found at {root}", flush=True)
            return root
    listing = "\n\n".join(f"{b}:\n{_tree(b)}" for b in CONFIG["search_roots"] if os.path.isdir(b))
    raise FileNotFoundError("MVTec AD not found. The search needs a folder that contains <category>/train/good, "
                            "for example .../bottle/train/good. What is under the search roots:\n" + listing +
                            "\nSet MVTEC_ROOT in the first cell to the folder that holds bottle/, cable/, ...")


def available_categories():
    root = find_mvtec_root()
    return [c for c in CONFIG["categories"] if os.path.isdir(os.path.join(root, c, "train", "good"))]


def list_images(folder):
    folder = Path(folder)
    if not folder.is_dir():
        return []
    return sorted(p for p in folder.iterdir() if p.suffix.lower() in IMAGE_EXT)


def read_rgb(path, size):
    image = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if image is None:
        raise IOError(f"cannot read {path}")
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    return cv2.resize(image, (size, size), interpolation=cv2.INTER_AREA)


def read_mask(path, size):
    if path is None or not os.path.exists(path):
        return np.zeros((size, size), np.uint8)
    mask = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    return (cv2.resize(mask, (size, size), interpolation=cv2.INTER_NEAREST) > 127).astype(np.uint8)


def load_category(category):
    root = Path(find_mvtec_root()) / category
    size = CONFIG["image_size"]
    rng = np.random.default_rng(stable_seed(CONFIG["seed"], category, "split"))

    train_paths = list_images(root / "train" / "good")
    order = rng.permutation(len(train_paths))
    if CONFIG["max_train_images"]:
        order = order[:CONFIG["max_train_images"]]
    n_cal = max(2, int(round(len(order) * CONFIG["calibration_fraction"])))
    cal_paths = [train_paths[i] for i in sorted(order[:n_cal])]
    fit_paths = [train_paths[i] for i in sorted(order[n_cal:])]

    items = []
    for defect_dir in sorted(p for p in (root / "test").iterdir() if p.is_dir()):
        for path in list_images(defect_dir):
            mask_path = None if defect_dir.name == "good" else root / "ground_truth" / defect_dir.name / f"{path.stem}_mask.png"
            items.append((path, defect_dir.name, mask_path))
    limit = CONFIG["max_test_images"]
    if limit and len(items) > limit:
        good = [i for i, it in enumerate(items) if it[1] == "good"]
        bad = [i for i, it in enumerate(items) if it[1] != "good"]
        n_good = min(len(good), max(1, limit // 3))
        keep = list(rng.choice(good, n_good, replace=False)) + list(rng.choice(bad, min(len(bad), limit - n_good), replace=False))
        items = [items[i] for i in sorted(keep)]

    return dict(
        category=category,
        fit=np.stack([read_rgb(p, size) for p in fit_paths]),
        cal=np.stack([read_rgb(p, size) for p in cal_paths]),
        test=np.stack([read_rgb(p, size) for p, _, _ in items]),
        test_masks=np.stack([read_mask(m, size) for _, _, m in items]),
        labels=np.array([0 if d == "good" else 1 for _, d, _ in items]),
        defects=[d for _, d, _ in items],
        test_paths=[str(p) for p, _, _ in items],
    )

In [4]:
def _gamma(image, mask, level, rng):
    gamma = level ** rng.choice([-1.0, 1.0])
    table = np.clip(255.0 * (np.arange(256) / 255.0) ** gamma, 0, 255).astype(np.uint8)
    return cv2.LUT(image, table), mask


def _color_temperature(image, mask, level, rng):
    shift = level * rng.choice([-1.0, 1.0])
    out = image.astype(np.float32)
    out[..., 0] *= 1.0 + shift
    out[..., 2] *= 1.0 - shift
    return np.clip(out, 0, 255).astype(np.uint8), mask


def _gaussian_noise(image, mask, level, rng):
    noise = rng.normal(0.0, level, image.shape)
    return np.clip(image.astype(np.float32) + noise, 0, 255).astype(np.uint8), mask


def _motion_blur(image, mask, level, rng):
    length = int(level)
    kernel = np.zeros((length, length), np.float32)
    angle = rng.uniform(0, np.pi)
    c = (length - 1) / 2
    dx, dy = np.cos(angle) * c, np.sin(angle) * c
    cv2.line(kernel, (int(round(c - dx)), int(round(c - dy))), (int(round(c + dx)), int(round(c + dy))), 1.0, 1)
    kernel /= kernel.sum()
    return cv2.filter2D(image, -1, kernel, borderType=cv2.BORDER_REFLECT_101), mask


def _jpeg(image, mask, level, rng):
    ok, buffer = cv2.imencode(".jpg", cv2.cvtColor(image, cv2.COLOR_RGB2BGR), [cv2.IMWRITE_JPEG_QUALITY, int(level)])
    return cv2.cvtColor(cv2.imdecode(buffer, cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB), mask


def _warp(image, mask, matrix, perspective):
    size = image.shape[0]
    warp = cv2.warpPerspective if perspective else cv2.warpAffine
    out = warp(image, matrix, (size, size), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT_101)
    if mask is not None:
        mask = warp(mask, matrix, (size, size), flags=cv2.INTER_NEAREST, borderMode=cv2.BORDER_CONSTANT, borderValue=0)
    return out, mask


def _rotation_matrix(size, level, rng):
    return cv2.getRotationMatrix2D((size / 2 - 0.5, size / 2 - 0.5), level * rng.choice([-1.0, 1.0]), 1.0)


def _perspective_matrix(size, level, rng):
    source = np.float32([[0, 0], [size - 1, 0], [size - 1, size - 1], [0, size - 1]])
    target = source + rng.uniform(-level, level, (4, 2)).astype(np.float32) * size
    return cv2.getPerspectiveTransform(source, target)


def _rotation(image, mask, level, rng):
    return _warp(image, mask, _rotation_matrix(image.shape[0], level, rng), False)


def _perspective(image, mask, level, rng):
    return _warp(image, mask, _perspective_matrix(image.shape[0], level, rng), True)


NUISANCES = {"gamma": _gamma, "color_temperature": _color_temperature, "gaussian_noise": _gaussian_noise,
             "motion_blur": _motion_blur, "jpeg": _jpeg, "rotation": _rotation, "perspective": _perspective}


def apply_nuisance(image, mask, family, severity, key):
    if family == "clean":
        return image, mask
    level = SEVERITY_TABLE[family][severity - 1]
    return NUISANCES[family](image, mask, level, np.random.default_rng(stable_seed(key)))


def valid_region(family, severity, key, size):
    if FAMILY_GROUP.get(family) != "geometric":
        return None
    level = SEVERITY_TABLE[family][severity - 1]
    rng = np.random.default_rng(stable_seed(key))
    ones = np.ones((size, size), np.uint8)
    if family == "rotation":
        valid = cv2.warpAffine(ones, _rotation_matrix(size, level, rng), (size, size), flags=cv2.INTER_NEAREST, borderValue=0)
    else:
        valid = cv2.warpPerspective(ones, _perspective_matrix(size, level, rng), (size, size), flags=cv2.INTER_NEAREST, borderValue=0)
    return {m: cv2.erode(valid, np.ones((2 * m + 1, 2 * m + 1), np.uint8)) for m in CONFIG["valid_margins"]}


def active_families(category):
    families = CONFIG["seen_families"] + CONFIG["heldout_families"]
    if category in CONFIG["geometry_semantic"]:
        families = [f for f in families if FAMILY_GROUP[f] != "geometric"]
    return families


def shift_set(category, split, images, masks, family, severity):
    if family == "clean":
        return images, masks, None
    out_images, out_masks, out_valid = [], [], []
    for i in range(len(images)):
        key = (CONFIG["seed"], category, split, i, family, severity)
        image, mask = apply_nuisance(images[i], None if masks is None else masks[i], family, severity, key)
        out_images.append(image)
        out_masks.append(mask)
        out_valid.append(valid_region(family, severity, key, images.shape[1]))
    valid = None if out_valid[0] is None else {m: np.stack([v[m] for v in out_valid]) for m in out_valid[0]}
    return np.stack(out_images), (None if masks is None else np.stack(out_masks)), valid


def augmented_training_set(category, images, seen):
    out = []
    for family in seen:
        for i in range(len(images)):
            severity = CONFIG["severities"][stable_seed(CONFIG["seed"], category, "aug", i, family) % len(CONFIG["severities"])]
            out.append(apply_nuisance(images[i], None, family, severity, (CONFIG["seed"], category, "fit", i, family, severity))[0])
    return np.stack(out) if out else np.zeros((0,) + images.shape[1:], np.uint8)


_CLAHE = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))


def photonorm(image):
    x = image.astype(np.float32)
    means = x.reshape(-1, 3).mean(0) + 1e-6
    x = np.clip(x * (means.mean() / means), 0, 255).astype(np.uint8)
    lab = cv2.cvtColor(x, cv2.COLOR_RGB2LAB)
    lab[..., 0] = _CLAHE.apply(lab[..., 0])
    return cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)


_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
_STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)


def crop_offset():
    return (CONFIG["image_size"] - CONFIG["crop_size"]) // 2


def to_tensor(images):
    o, c = crop_offset(), CONFIG["crop_size"]
    x = torch.from_numpy(np.ascontiguousarray(images[:, o:o + c, o:o + c, :])).to(DEVICE)
    x = x.permute(0, 3, 1, 2).float() / 255.0
    return (x - _MEAN.to(DEVICE)) / _STD.to(DEVICE)


def masks_for_eval(masks, valid=None):
    o, c = crop_offset(), CONFIG["crop_size"]
    m = torch.from_numpy(np.ascontiguousarray(masks[:, o:o + c, o:o + c])).float()[:, None]
    m = F.adaptive_max_pool2d(m, CONFIG["pixel_eval_size"])
    if valid is None:
        return m[:, 0].numpy().astype(np.uint8), None
    # min-pool: a low-res pixel is valid only if all of it is real content
    v = torch.from_numpy(np.ascontiguousarray(valid[:, o:o + c, o:o + c])).float()[:, None]
    v = 1.0 - F.adaptive_max_pool2d(1.0 - v, CONFIG["pixel_eval_size"])
    return m[:, 0].numpy().astype(np.uint8), v[:, 0].numpy() > 0.5

In [5]:
# backbones

class WRNTrunk(nn.Module):
    def __init__(self, pretrained):
        super().__init__()
        weights = torchvision.models.Wide_ResNet50_2_Weights.IMAGENET1K_V1 if pretrained else None
        net = torchvision.models.wide_resnet50_2(weights=weights)
        self.stem = nn.Sequential(net.conv1, net.bn1, net.relu, net.maxpool)
        self.layer1, self.layer2, self.layer3 = net.layer1, net.layer2, net.layer3

    def forward(self, x):
        l1 = self.layer1(self.stem(x))
        l2 = self.layer2(l1)
        return l1, l2, self.layer3(l2)


class PatchCoreWRNHead(nn.Module):
    def forward(self, l2, l3):
        l2 = F.avg_pool2d(l2, 3, 1, 1)
        l3 = F.avg_pool2d(l3, 3, 1, 1)
        l3 = F.interpolate(l3, size=l2.shape[-2:], mode="bilinear", align_corners=False)
        b, _, h, w = l2.shape
        l2 = l2.reshape(b, 256, 2, h, w).mean(2)
        l3 = l3.reshape(b, 256, 4, h, w).mean(2)
        return torch.cat([l2, l3], 1)


class PatchCoreWRN(nn.Module):
    def __init__(self, trunk):
        super().__init__()
        self.trunk, self.head = trunk, PatchCoreWRNHead()

    def forward(self, x):
        _, l2, l3 = self.trunk(x)
        return self.head(l2, l3)


class DinoTrunk(nn.Module):
    def __init__(self, name, pretrained, layers):
        super().__init__()
        import timm
        self.model = timm.create_model(name, pretrained=pretrained, img_size=CONFIG["crop_size"])
        self.layers = tuple(layers)
        self._store = {}
        for index in self.layers:
            self.model.blocks[index].register_forward_hook(self._hook(index))

    def _hook(self, index):
        def fn(module, inputs, output):
            self._store[index] = output
        return fn

    def forward(self, x):
        self._store.clear()
        self.model.forward_features(x)
        gh, gw = self.model.patch_embed.grid_size
        prefix = self.model.num_prefix_tokens
        maps = []
        for index in self.layers:
            tokens = self._store[index][:, prefix:, :]
            tokens = F.layer_norm(tokens, (tokens.shape[-1],))
            grid = tokens.transpose(1, 2).reshape(x.shape[0], -1, gh, gw)
            maps.append(F.avg_pool2d(grid, 3, 1, 1))
        return torch.cat(maps, 1)


def padim_features(l1, l2, l3, index):
    size = l1.shape[-2:]
    l2 = F.interpolate(l2, size=size, mode="bilinear", align_corners=False)
    l3 = F.interpolate(l3, size=size, mode="bilinear", align_corners=False)
    return torch.cat([l1, l2, l3], 1).index_select(1, index)


def load_models():
    models = {"wrn": WRNTrunk(CONFIG["pretrained"]).to(DEVICE).eval(), "pc_head": PatchCoreWRNHead().to(DEVICE)}
    generator = torch.Generator().manual_seed(CONFIG["seed"])
    models["padim_index"] = torch.randperm(256 + 512 + 1024, generator=generator)[:CONFIG["padim_dims"]].sort().values.to(DEVICE)
    models["dino"] = None
    if any(DETECTOR_SPECS[d]["view"] == "dino" for d in CONFIG["detectors"]):
        try:
            models["dino"] = DinoTrunk(CONFIG["dino_name"], CONFIG["pretrained"], CONFIG["dino_layers"]).to(DEVICE).eval()
        except Exception as error:
            log(f"DINOv2 could not be loaded, its detectors are skipped: {error}")
    return models


def active_specs(models):
    specs = {d: DETECTOR_SPECS[d] for d in CONFIG["detectors"]}
    if models["dino"] is None:
        specs = {d: s for d, s in specs.items() if s["view"] != "dino"}
    return specs


@torch.inference_mode()
def view_features(models, view, images, need_padim):
    if view == "wrn_photo":
        images = np.stack([photonorm(image) for image in images])
    x = to_tensor(images)
    if view == "dino":
        return {"pc": models["dino"](x)}
    l1, l2, l3 = models["wrn"](x)
    out = {"pc": models["pc_head"](l2, l3)}
    if need_padim:
        out["padim"] = padim_features(l1, l2, l3, models["padim_index"])
    return out

In [6]:
# detectors

@torch.inference_mode()
def greedy_coreset(features, n_select, projection_dim, seed):
    n = features.shape[0]
    if n_select >= n:
        return torch.arange(n)
    generator = torch.Generator().manual_seed(seed)
    projection = (torch.randn(features.shape[1], projection_dim, generator=generator) / math.sqrt(projection_dim)).to(DEVICE)
    z = torch.empty(n, projection_dim, device=DEVICE)
    for start in range(0, n, 65536):
        z[start:start + 65536] = features[start:start + 65536].to(DEVICE).float() @ projection
    squared = (z * z).sum(1)
    current = torch.randint(n, (1,), generator=generator).to(DEVICE)[0]
    min_distance = (squared + squared[current] - 2 * (z @ z[current])).clamp_min(0)
    chosen = [current]
    for _ in range(n_select - 1):
        current = torch.argmax(min_distance)
        chosen.append(current)
        min_distance = torch.minimum(min_distance, (squared + squared[current] - 2 * (z @ z[current])).clamp_min(0))
    return torch.stack(chosen).cpu()


class PatchCore:
    def __init__(self, bank_size, seed):
        self.bank_size, self.seed = bank_size, seed

    def fit(self, features):
        index = greedy_coreset(features, self.bank_size, CONFIG["coreset_projection"], self.seed)
        self.bank = features[index].float().to(DEVICE)
        self.bank_sq = (self.bank * self.bank).sum(1)
        return self

    @torch.inference_mode()
    def patch_scores(self, features):
        b, d, h, w = features.shape
        query = features.permute(0, 2, 3, 1).reshape(-1, d).float()
        out = []
        for chunk in query.split(4096):
            d2 = (chunk * chunk).sum(1, keepdim=True) + self.bank_sq[None] - 2 * chunk @ self.bank.T
            out.append(d2.clamp_min(0).min(1).values.sqrt())
        return torch.cat(out).reshape(b, h, w)


class PaDiMStats:
    def __init__(self):
        self.n, self.total, self.outer = 0, None, None

    @torch.inference_mode()
    def add(self, features):
        b, d, h, w = features.shape
        x = features.permute(0, 2, 3, 1).reshape(b, h * w, d).double()
        if self.total is None:
            self.total = torch.zeros(h * w, d, dtype=torch.float64, device=x.device)
            self.outer = torch.zeros(h * w, d, d, dtype=torch.float64, device=x.device)
            self.shape = (h, w)
        self.n += b
        self.total += x.sum(0)
        self.outer += torch.einsum("bpi,bpj->pij", x, x)

    def copy(self):
        other = PaDiMStats()
        other.n, other.shape = self.n, self.shape
        other.total, other.outer = self.total.clone(), self.outer.clone()
        return other


class PaDiM:
    def fit(self, stats):
        mean = stats.total / stats.n
        cov = (stats.outer - stats.n * torch.einsum("pi,pj->pij", mean, mean)) / max(stats.n - 1, 1)
        cov = cov + CONFIG["padim_eps"] * torch.eye(cov.shape[-1], dtype=cov.dtype, device=cov.device)
        self.mean, self.inverse, self.shape = mean.float(), torch.linalg.inv(cov).float(), stats.shape
        return self

    @torch.inference_mode()
    def patch_scores(self, features):
        b, d, h, w = features.shape
        x = features.permute(0, 2, 3, 1).reshape(b, h * w, d) - self.mean[None]
        m = torch.einsum("bpi,pij,bpj->bp", x, self.inverse, x)
        return m.clamp_min(0).sqrt().reshape(b, h, w)


def gaussian_kernel(sigma):
    radius = int(math.ceil(4 * sigma))
    t = torch.arange(-radius, radius + 1, dtype=torch.float32)
    k = torch.exp(-t ** 2 / (2 * sigma ** 2))
    return k / k.sum(), radius


@torch.inference_mode()
def grid_to_map(grid):
    size = CONFIG["crop_size"]
    x = F.interpolate(grid[:, None].float(), size=(size, size), mode="bilinear", align_corners=False)
    k, r = gaussian_kernel(CONFIG["blur_sigma"])
    k = k.to(x.device)
    x = F.conv2d(F.pad(x, (r, r, 0, 0), mode="reflect"), k.view(1, 1, 1, -1))
    x = F.conv2d(F.pad(x, (0, 0, r, r), mode="reflect"), k.view(1, 1, -1, 1))
    return x[:, 0]


@torch.inference_mode()
def eval_map(grid):
    return F.interpolate(grid_to_map(grid)[:, None], size=CONFIG["pixel_eval_size"], mode="area")[:, 0]


@torch.inference_mode()
def collect_features(models, view, images, need_pc, need_padim, stats=None):
    patches = []
    if need_padim and stats is None:
        stats = PaDiMStats()
    for start in range(0, len(images), CONFIG["batch_size"]):
        feats = view_features(models, view, images[start:start + CONFIG["batch_size"]], need_padim)
        if need_pc:
            f = feats["pc"]
            patches.append(f.permute(0, 2, 3, 1).reshape(-1, f.shape[1]).half().cpu())
        if need_padim:
            stats.add(feats["padim"])
    return (torch.cat(patches) if patches else None), (stats if need_padim else None)


def fit_detectors(category, data, seen, specs, models):
    augmented = augmented_training_set(category, data["fit"], seen)
    detectors, info = {}, {}
    for view in sorted({s["view"] for s in specs.values()}):
        group = {d: s for d, s in specs.items() if s["view"] == view}
        need_pc = any(s["kind"] == "patchcore" for s in group.values())
        need_padim = any(s["kind"] == "padim" for s in group.values())
        need_aug = any(s["aug"] for s in group.values()) and len(augmented) > 0
        clean_pc, clean_stats = collect_features(models, view, data["fit"], need_pc, need_padim)
        extra_pc, aug_stats = None, clean_stats
        if need_aug:
            extra_pc, aug_stats = collect_features(models, view, augmented, need_pc, need_padim,
                                                   clean_stats.copy() if clean_stats is not None else None)
        for name, spec in group.items():
            if spec["kind"] == "patchcore":
                bank_size = min(int(math.ceil(CONFIG["coreset_fraction"] * len(clean_pc))), CONFIG["coreset_cap"])
                pool = torch.cat([clean_pc, extra_pc]) if (spec["aug"] and extra_pc is not None) else clean_pc
                detectors[name] = PatchCore(bank_size, stable_seed(CONFIG["seed"], category, name)).fit(pool)
                info[name] = dict(pool=int(len(pool)), bank=int(bank_size))
            else:
                stats = aug_stats if spec["aug"] else clean_stats
                detectors[name] = PaDiM().fit(stats)
                info[name] = dict(pool=int(stats.n), bank=0)
        del clean_pc, extra_pc
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    return detectors, info


@torch.inference_mode()
def _masked_max(values, valid):
    masked = torch.where(valid > 0.5, values, torch.full_like(values, -float("inf"))).flatten(1).max(1).values
    plain = values.flatten(1).max(1).values
    return torch.where(torch.isfinite(masked), masked, plain)


@torch.inference_mode()
def _masked_median(values, valid):
    filled = torch.where(valid > 0.5, values, torch.full_like(values, float("nan"))).flatten(1)
    med = torch.nanmedian(filled, dim=1).values
    return torch.where(torch.isfinite(med), med, values.flatten(1).median(1).values)


@torch.inference_mode()
def score_images(models, specs, detectors, images, want_maps, valid=None):
    views = {}
    for spec in specs.values():
        views[spec["view"]] = views.get(spec["view"], False) or spec["kind"] == "padim"
    margins = CONFIG["valid_margins"]
    out = {d: {"scores": [], "scores_local": [], "maps": [], "valid": {m: [] for m in margins}} for d in specs}
    o, c = crop_offset(), CONFIG["crop_size"]
    for start in range(0, len(images), CONFIG["batch_size"]):
        batch = images[start:start + CONFIG["batch_size"]]
        v224 = None
        if valid is not None:
            v224 = {m: torch.from_numpy(np.ascontiguousarray(valid[m][start:start + CONFIG["batch_size"], o:o + c, o:o + c]))
                    .float().to(DEVICE)[:, None] for m in margins}
        for view, need_padim in views.items():
            feats = view_features(models, view, batch, need_padim)
            for name, spec in specs.items():
                if spec["view"] != view:
                    continue
                grid = detectors[name].patch_scores(feats["padim"] if spec["kind"] == "padim" else feats["pc"])
                values = grid_to_map(grid) if spec["kind"] == "padim" else grid
                score = values.flatten(1).max(1).values
                masks = {}
                for m in margins:
                    if v224 is None:
                        masks[m] = torch.ones_like(values)
                    elif spec["kind"] == "padim":
                        masks[m] = v224[m][:, 0]
                    else:
                        masks[m] = 1.0 - F.adaptive_max_pool2d(1.0 - v224[m], grid.shape[-2:])[:, 0]
                    out[name]["valid"][m].append((score if v224 is None else _masked_max(values, masks[m])).cpu().numpy())
                main = margins[0]
                score_local = _masked_max(values, masks[main]) - _masked_median(values, masks[main])
                out[name]["scores"].append(score.cpu().numpy())
                out[name]["scores_local"].append(score_local.cpu().numpy())
                if want_maps:
                    out[name]["maps"].append(eval_map(grid).cpu().numpy())
    for name in out:
        out[name]["scores"] = np.concatenate(out[name]["scores"])
        out[name]["scores_local"] = np.concatenate(out[name]["scores_local"])
        out[name]["valid"] = {m: np.concatenate(v) for m, v in out[name]["valid"].items()}
        out[name]["scores_valid"] = out[name]["valid"][margins[0]]
        out[name]["maps"] = np.concatenate(out[name]["maps"]) if want_maps else None
    return out

In [7]:
# metrics and stats helpers

def auroc(negatives, positives):
    negatives = np.asarray(negatives, float).ravel()
    positives = np.asarray(positives, float).ravel()
    if len(negatives) == 0 or len(positives) == 0:
        return np.nan
    ranks = rankdata(np.concatenate([negatives, positives]))
    n_pos, n_neg = len(positives), len(negatives)
    return float((ranks[n_neg:].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg))


def pixel_auroc(maps, masks, valid=None):
    keep = np.ones(masks.shape, bool) if valid is None else valid
    return auroc(maps[(masks == 0) & keep], maps[(masks == 1) & keep])


def aupro(maps, masks, valid=None, fpr_limit=0.3, n_thresholds=200):
    if valid is not None:
        masks = masks * valid
        maps = np.where(valid, maps, np.nan)
    background = maps[masks == 0]
    background = np.sort(background[~np.isnan(background)])
    regions = []
    structure = np.ones((3, 3), int)
    for score_map, mask in zip(maps, masks):
        if mask.any():
            labelled, count = ndimage.label(mask, structure=structure)
            for r in range(1, count + 1):
                regions.append(np.sort(score_map[labelled == r]))
    if not regions or len(background) == 0:
        return np.nan
    targets = np.linspace(0.0, fpr_limit, n_thresholds)
    thresholds = np.quantile(background, 1.0 - targets)
    fpr = 1.0 - np.searchsorted(background, thresholds, side="right") / len(background)
    pro = np.array([np.mean([1.0 - np.searchsorted(r, t, side="right") / len(r) for r in regions]) for t in thresholds])
    order = np.argsort(fpr, kind="stable")
    fpr, pro = fpr[order], pro[order]
    grid = np.concatenate([[0.0], fpr[fpr < fpr_limit], [fpr_limit]])
    values = np.interp(grid, fpr, pro)
    integrate = getattr(np, "trapezoid", None) or np.trapz
    return float(integrate(values, grid) / fpr_limit)


def conformal_threshold(calibration_scores, alpha):
    s = np.sort(np.asarray(calibration_scores, float))
    n = len(s)
    k = int(math.ceil((n + 1) * (1 - alpha)))
    return float("inf") if k > n else float(s[k - 1])


def bootstrap_mean_ci(values, n_boot, seed=0, level=0.95):
    values = np.asarray(values, float)
    values = values[~np.isnan(values)]
    if len(values) < 2:
        return (np.nan, np.nan)
    rng = np.random.default_rng(seed)
    means = values[rng.integers(0, len(values), (n_boot, len(values)))].mean(1)
    return (float(np.quantile(means, (1 - level) / 2)), float(np.quantile(means, 1 - (1 - level) / 2)))


def paired_wilcoxon(differences, alternative):
    d = np.asarray(differences, float)
    d = d[~np.isnan(d)]
    if len(d) == 0 or np.all(d == 0):
        return np.nan
    return float(wilcoxon(d, alternative=alternative, zero_method="wilcox").pvalue)


def holm(pvalues):
    p = np.asarray(pvalues, float)
    out = np.full_like(p, np.nan)
    valid = np.where(~np.isnan(p))[0]
    order = valid[np.argsort(p[valid])]
    running = 0.0
    for rank, i in enumerate(order):
        running = max(running, min(1.0, (len(valid) - rank) * p[i]))
        out[i] = running
    return out


def regime_of(family):
    if family == "clean":
        return "clean"
    return "seen" if family in CONFIG["seen_families"] else "heldout"

In [8]:
def record_columns():
    extra = [f"score_valid_m{m}" for m in CONFIG["valid_margins"][1:]]
    return ["category", "detector", "family", "severity", "split", "index", "label", "defect", "score", "score_valid",
            "score_local", "defect_area"] + extra


def thumbnail(image, size):
    o, c = crop_offset(), CONFIG["crop_size"]
    return cv2.resize(image[o:o + c, o:o + c], (size, size), interpolation=cv2.INTER_AREA)


def update_gallery(gallery, scores, maps, images, data, family, severity):
    k = CONFIG["gallery_k"]
    if family == "clean":
        pool, candidates = "misses", np.where(data["labels"] == 1)[0]
        keys = -scores[candidates]
    else:
        pool, candidates = "false_alarms", np.where(data["labels"] == 0)[0]
        keys = scores[candidates]
    for j in np.argsort(-keys)[:k]:
        i = candidates[j]
        gallery[pool].append(dict(key=float(keys[j]), score=float(scores[i]), family=family, severity=int(severity),
                                  index=int(i), defect=data["defects"][i],
                                  image=thumbnail(images[i], CONFIG["pixel_eval_size"]), map=maps[i].astype(np.float32)))
    gallery[pool] = sorted(gallery[pool], key=lambda g: -g["key"])[:k]


def augmented_pick(category, tag, image, seen):
    choices = ["clean"] + list(seen)
    family = choices[stable_seed(CONFIG["seed"], category, tag, image) % len(choices)]
    severity = 0 if family == "clean" else CONFIG["severities"][stable_seed(CONFIG["seed"], category, tag, image, family) % len(CONFIG["severities"])]
    return family, severity


def compute_thresholds(records, category, seen):
    thresholds = {}
    cal = records[records.split == "cal"]
    for name, group in cal.groupby("detector"):
        lookup = {(f, int(s), int(i)): v for f, s, i, v in zip(group.family, group.severity, group["index"], group.score)}
        indices = sorted(group[group.family == "clean"]["index"].astype(int).unique())
        clean = np.array([lookup[("clean", 0, i)] for i in indices])
        mixed = np.array([lookup.get((*augmented_pick(category, "calpool", i, seen), i), lookup[("clean", 0, i)]) for i in indices])
        thresholds[name] = dict(clean=conformal_threshold(clean, CONFIG["alpha"]),
                                aug=conformal_threshold(mixed, CONFIG["alpha"]),
                                n_clean=int(len(clean)), n_aug=int(len(mixed)))
    return thresholds


def category_metrics(category, records, pixel, thresholds):
    rows = []
    test, cal = records[records.split == "test"], records[records.split == "cal"]
    for name, group in test.groupby("detector"):
        clean = group[group.family == "clean"]
        clean_normal = clean[clean.label == 0].score.values
        clean_anomaly = clean[clean.label == 1].score.values
        clean_normal_valid = clean[clean.label == 0].score_valid.values
        cal_det = cal[cal.detector == name]
        cal_clean_median = float(np.median(cal_det[cal_det.family == "clean"].score))
        for (family, severity), part in group.groupby(["family", "severity"]):
            normal = part[part.label == 0].score.values
            anomaly = part[part.label == 1].score.values
            colour = part[(part.label == 1) & part.defect.str.contains("color")].score.values
            cal_part = cal_det[(cal_det.family == family) & (cal_det.severity == severity)].score.values
            row = dict(category=category, detector=name, family=family, severity=int(severity), regime=regime_of(family),
                       group=FAMILY_GROUP.get(family, "clean"),
                       img_auroc=auroc(normal, anomaly),
                       img_auroc_valid=auroc(part[part.label == 0].score_valid.values, part[part.label == 1].score_valid.values),
                       **{f"img_auroc_valid_m{m}": auroc(part[part.label == 0][f"score_valid_m{m}"].values,
                                                         part[part.label == 1][f"score_valid_m{m}"].values)
                          for m in CONFIG["valid_margins"][1:]},
                       img_auroc_local=auroc(part[part.label == 0].score_local.values, part[part.label == 1].score_local.values),
                       conf_auroc_local=auroc(part[part.label == 0].score_local.values, clean[clean.label == 1].score_local.values),
                       nuis_auroc_valid=auroc(clean_normal_valid, part[part.label == 0].score_valid.values) if family != "clean" else np.nan,
                       conf_auroc=auroc(normal, clean_anomaly),
                       nuis_auroc=auroc(clean_normal, normal) if family != "clean" else np.nan,
                       color_auroc=auroc(normal, colour) if len(colour) else np.nan,
                       ndr=float(np.median(cal_part) / cal_clean_median - 1.0) if len(cal_part) else np.nan,
                       n_normal=int(len(normal)), n_anomaly=int(len(anomaly)))
            for policy in ("clean", "aug"):
                t = thresholds[name][policy]
                row[f"fa_{policy}"] = int((normal > t).sum())
                row[f"tp_{policy}"] = int((anomaly > t).sum())
                row[f"fpr_{policy}"] = float((normal > t).mean()) if len(normal) else np.nan
                row[f"tpr_{policy}"] = float((anomaly > t).mean()) if len(anomaly) else np.nan
            rows.append(row)
    metrics = pd.DataFrame(rows)
    return metrics.merge(pixel, on=["category", "detector", "family", "severity"], how="left")


FINGERPRINT_KEYS = ["image_size", "crop_size", "pixel_eval_size", "blur_sigma", "calibration_fraction",
                    "max_train_images", "max_test_images", "coreset_fraction", "coreset_cap", "coreset_projection",
                    "padim_dims", "padim_eps", "valid_margins", "dino_name", "dino_layers", "pretrained", "alpha",
                    "seed", "seen_families", "heldout_families", "severities", "geometry_semantic", "detectors"]


def config_fingerprint():
    payload = {k: CONFIG[k] for k in FINGERPRINT_KEYS}
    payload["pipeline_version"] = PIPELINE_VERSION
    return hashlib.sha1(json.dumps(payload, sort_keys=True, default=str).encode()).hexdigest()[:12]


def run_category(category, models):
    out_dir = Path(CONFIG["output_dir"]) / "categories" / category
    marker = out_dir / "metrics.csv"
    stamp = out_dir / "fingerprint.txt"
    if marker.exists() and not CONFIG["overwrite"]:
        saved = stamp.read_text().strip() if stamp.exists() else None
        if saved == config_fingerprint():
            log(f"{category}: loaded from checkpoint")
            return pd.read_csv(marker)
        log(f"{category}: checkpoint was made with different settings, recomputing")
    out_dir.mkdir(parents=True, exist_ok=True)
    started = time.time()
    set_seed(stable_seed(CONFIG["seed"], category))
    data = load_category(category)
    families = active_families(category)
    seen = [f for f in CONFIG["seen_families"] if f in families]
    heldout = [f for f in CONFIG["heldout_families"] if f in families]
    conditions = [("clean", 0)] + [(f, s) for f in seen + heldout for s in CONFIG["severities"]]
    specs = active_specs(models)
    log(f"{category}: fit {len(data['fit'])}, cal {len(data['cal'])}, test {len(data['test'])} "
        f"({int(data['labels'].sum())} anomalous), {len(conditions)} conditions, {len(specs)} detectors")

    detectors, fit_info = fit_detectors(category, data, seen, specs, models)
    log(f"{category}: detectors fitted in {time.time() - started:.0f}s")

    records, pixel_rows = [], []
    gallery = {"false_alarms": [], "misses": []}
    n_cal, n_test = len(data["cal"]), len(data["test"])
    o, c = crop_offset(), CONFIG["crop_size"]
    area = data["test_masks"][:, o:o + c, o:o + c].reshape(n_test, -1).mean(1)
    for family, severity in conditions:
        cal_images, _, cal_valid = shift_set(category, "cal", data["cal"], None, family, severity)
        test_images, test_masks, test_valid = shift_set(category, "test", data["test"], data["test_masks"], family, severity)
        cal_out = score_images(models, specs, detectors, cal_images, want_maps=False, valid=cal_valid)
        test_out = score_images(models, specs, detectors, test_images, want_maps=True, valid=test_valid)
        margins = CONFIG["valid_margins"]
        eval_valid = {}
        for m in margins:
            eval_masks, eval_valid[m] = masks_for_eval(test_masks, None if test_valid is None else test_valid[m])
        for name in specs:
            extra_cal = [cal_out[name]["valid"][m] for m in margins[1:]]
            extra_test = [test_out[name]["valid"][m] for m in margins[1:]]
            for i in range(n_cal):
                records.append((category, name, family, severity, "cal", i, 0, "good", float(cal_out[name]["scores"][i]),
                                float(cal_out[name]["scores_valid"][i]), float(cal_out[name]["scores_local"][i]), 0.0,
                                *[float(v[i]) for v in extra_cal]))
            for i in range(n_test):
                records.append((category, name, family, severity, "test", i, int(data["labels"][i]), data["defects"][i],
                                float(test_out[name]["scores"][i]), float(test_out[name]["scores_valid"][i]),
                                float(test_out[name]["scores_local"][i]), float(area[i]), *[float(v[i]) for v in extra_test]))
            maps = test_out[name]["maps"]
            row = dict(category=category, detector=name, family=family, severity=severity,
                       pix_auroc=pixel_auroc(maps, eval_masks, eval_valid[margins[0]]),
                       aupro=aupro(maps, eval_masks, eval_valid[margins[0]]))
            for m in margins[1:]:
                if test_valid is None:
                    row[f"pix_auroc_m{m}"], row[f"aupro_m{m}"] = row["pix_auroc"], row["aupro"]
                else:
                    row[f"pix_auroc_m{m}"] = pixel_auroc(maps, eval_masks, eval_valid[m])
                    row[f"aupro_m{m}"] = aupro(maps, eval_masks, eval_valid[m])
            pixel_rows.append(row)
        if CONFIG["gallery_detector"] in specs:
            g = test_out[CONFIG["gallery_detector"]]
            update_gallery(gallery, g["scores"], g["maps"], test_images, data, family, severity)

    records = pd.DataFrame(records, columns=record_columns())
    thresholds = compute_thresholds(records, category, seen)
    metrics = category_metrics(category, records, pd.DataFrame(pixel_rows), thresholds)

    records.to_csv(out_dir / "records.csv.gz", index=False)
    with open(out_dir / "thresholds.json", "w") as handle:
        json.dump(dict(thresholds=thresholds, fit=fit_info, seen=seen, heldout=heldout), handle, indent=1)
    np.savez_compressed(out_dir / "gallery.npz", **{f"{pool}_{j}_{key}": item[key] for pool in gallery
                                                     for j, item in enumerate(gallery[pool]) for key in ("image", "map")})
    with open(out_dir / "gallery.json", "w") as handle:
        json.dump({pool: [{k: v for k, v in item.items() if k not in ("image", "map")} for item in items]
                   for pool, items in gallery.items()}, handle, indent=1)
    if category == CONFIG["deploy_category"] and "patchcore_wrn" in detectors:
        save_deployment_bank(category, data, detectors["patchcore_wrn"], thresholds["patchcore_wrn"], records)
    metrics.to_csv(marker, index=False)
    stamp.write_text(config_fingerprint())
    log(f"{category}: done in {time.time() - started:.0f}s")
    del detectors
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return metrics


def save_deployment_bank(category, data, detector, threshold, records):
    folder = Path(CONFIG["output_dir"]) / "deploy"
    folder.mkdir(parents=True, exist_ok=True)
    detector.bank.cpu().numpy().astype(np.float32).tofile(folder / "bank.bin")
    clean = records[(records.detector == "patchcore_wrn") & (records.family == "clean") & (records.split == "test")]
    anomalous = np.where(data["labels"] == 1)[0]
    normal = np.where(data["labels"] == 0)[0]
    refs = {}
    for tag, pool in (("anomalous", anomalous), ("normal", normal)):
        if len(pool):
            i = int(pool[0])
            refs[tag] = dict(image=data["test_paths"][i], python_score=float(clean[clean["index"] == i].score.iloc[0]))
    info = dict(category=category, bank_rows=int(detector.bank.shape[0]), dim=int(detector.bank.shape[1]),
                threshold=threshold["clean"], alpha=CONFIG["alpha"], references=refs)
    with open(folder / "deploy_info.json", "w") as handle:
        json.dump(info, handle, indent=1)

In [9]:
def per_category(metrics, value, **filters):
    d = metrics
    for column, wanted in filters.items():
        d = d[d[column].isin(wanted if isinstance(wanted, (list, tuple, set)) else [wanted])]
    return d.groupby(["category", "detector"])[value].mean().unstack("detector")


def summary_table(metrics):
    columns = ["img_auroc", "conf_auroc", "nuis_auroc", "pix_auroc", "aupro"]
    by_cat = metrics.groupby(["detector", "regime", "category"])[columns].mean()
    return by_cat.groupby(["detector", "regime"]).mean().round(4)


def augmentation_transfer_table(metrics):
    rows = []
    for base, aug in [("patchcore_wrn", "patchcore_wrn_aug"), ("patchcore_dino", "patchcore_dino_aug"), ("padim_wrn", "padim_wrn_aug")]:
        if base not in metrics.detector.values or aug not in metrics.detector.values:
            continue
        for group in ["photometric", "sensor", "geometric"]:
            for regime in ["seen", "heldout"]:
                table = per_category(metrics, "img_auroc", group=group, regime=regime)
                if base in table and aug in table:
                    diff = (table[aug] - table[base]).dropna()
                    if len(diff):
                        rows.append(dict(pair=f"{aug} - {base}", group=group, regime=regime, n=len(diff),
                                         mean_gain=round(float(diff.mean()), 4)))
    return pd.DataFrame(rows)


def load_records(categories):
    frames, thresholds = [], {}
    for category in categories:
        folder = Path(CONFIG["output_dir"]) / "categories" / category
        frames.append(pd.read_csv(folder / "records.csv.gz"))
        thresholds[category] = json.load(open(folder / "thresholds.json"))["thresholds"]
    records = pd.concat(frames, ignore_index=True)
    records["regime"] = records.family.map(regime_of)
    return records, thresholds


def _cluster_bootstrap_rate(hits, trials, clusters_by_category, n_boot, seed, resample_categories=True):
    rng = np.random.default_rng(seed)
    cats = list(clusters_by_category)
    rates = []
    for _ in range(n_boot):
        h = t = 0.0
        chosen = rng.choice(len(cats), len(cats), replace=True) if resample_categories else range(len(cats))
        for c in chosen:
            idx = clusters_by_category[cats[c]]
            pick = idx[rng.integers(0, len(idx), len(idx))]
            h += hits[pick].sum()
            t += trials[pick].sum()
        rates.append(h / t if t else np.nan)
    rates = np.array(rates)
    rates = rates[~np.isnan(rates)]
    if len(rates) == 0:
        return (np.nan, np.nan)
    return (float(np.quantile(rates, 0.025)), float(np.quantile(rates, 0.975)))


def train_image_level(records, thresholds):
    test = records[records.split == "test"]
    frames = []
    for policy in ("clean", "aug"):
        t = np.array([thresholds[c][d][policy] for c, d in zip(test.category.values, test.detector.values)])
        f = test.assign(hit=(test.score.values > t).astype(int))
        g = f.groupby(["detector", "regime", "category", "index", "label"]).hit.agg(["sum", "count"]).reset_index()
        frames.append(g.rename(columns={"sum": "hits", "count": "trials"}).assign(calibration=f"train-{policy}"))
    return pd.concat(frames, ignore_index=True)


def rate_table(image_level):
    rows = []
    n_boot = min(CONFIG["bootstrap"], 2000)
    for (name, regime, policy), g in image_level.groupby(["detector", "regime", "calibration"]):
        normal, anomaly = g[g.label == 0], g[g.label == 1]
        clusters = {c: np.where(normal.category.values == c)[0] for c in normal.category.unique()}
        two_stage = len(clusters) >= 5
        low, high = _cluster_bootstrap_rate(normal.hits.values.astype(float), normal.trials.values.astype(float),
                                            clusters, n_boot, stable_seed(name, regime, policy), two_stage)
        rows.append(dict(detector=name, regime=regime, calibration=policy, categories=len(clusters),
                         distinct_normals=int(len(normal)),
                         fpr=round(float(normal.hits.sum() / normal.trials.sum()), 4),
                         fpr_low=round(low, 4), fpr_high=round(high, 4),
                         tpr=round(float(anomaly.hits.sum() / anomaly.trials.sum()), 4) if len(anomaly) else np.nan,
                         ci_method="categories then images" if two_stage else "images, categories fixed"))
    return pd.DataFrame(rows)


def exchangeability_table(records):
    from scipy.stats import mannwhitneyu
    rows = []
    clean = records[records.family == "clean"]
    for (category, name), part in clean.groupby(["category", "detector"]):
        cal = part[part.split == "cal"].score.values
        test_normal = part[(part.split == "test") & (part.label == 0)].score.values
        if len(cal) and len(test_normal):
            rows.append(dict(category=category, detector=name, n_cal=len(cal), n_test_normal=len(test_normal),
                             split_auroc=auroc(cal, test_normal),
                             p=float(mannwhitneyu(test_normal, cal, alternative="two-sided").pvalue),
                             median_ratio=float(np.median(test_normal) / np.median(cal))))
    table = pd.DataFrame(rows)
    summary = table.groupby("detector").agg(categories=("category", "count"), median_split_auroc=("split_auroc", "median"),
                                            p_below_005_uncorrected=("p", lambda p: int((p < 0.05).sum())),
                                            median_ratio=("median_ratio", "median")).round(4)
    return table, summary


def test_split_calibration(records, n_splits=50, column="score"):
    detail, image_rows = [], []
    test = records[records.split == "test"]
    for (category, name), part in test.groupby(["category", "detector"]):
        normal = part[part.label == 0].pivot_table(index=["family", "severity"], columns="index", values=column)
        anomaly = part[part.label == 1].pivot_table(index=["family", "severity"], columns="index", values=column)
        if ("clean", 0) not in normal.index or normal.shape[1] < 4:
            continue
        anomaly = anomaly.reindex(normal.index)
        conds = list(normal.index)
        S, A = normal.values, anomaly.values
        images, n_img = np.array(normal.columns), normal.shape[1]
        clean_row = conds.index(("clean", 0))
        seen = [f for f in CONFIG["seen_families"] if any(c[0] == f for c in conds)]
        mixed_row = np.array([conds.index(augmented_pick(category, "tsaug", int(images[j]), seen))
                              if augmented_pick(category, "tsaug", int(images[j]), seen) in conds else clean_row
                              for j in range(n_img)])
        rng = np.random.default_rng(stable_seed(category, name, "testsplit"))
        hits = {p: np.zeros(S.shape) for p in ("clean", "aug")}
        a_hits = {p: np.zeros(A.shape) for p in ("clean", "aug")}
        trials = np.zeros(S.shape)
        for _ in range(n_splits):
            perm = rng.permutation(n_img)
            cal, ev = perm[:n_img // 2], perm[n_img // 2:]
            thresholds = {"clean": conformal_threshold(S[clean_row, cal], CONFIG["alpha"]),
                          "aug": conformal_threshold(S[mixed_row[cal], cal], CONFIG["alpha"])}
            trials[:, ev] += 1
            for p, t in thresholds.items():
                hits[p][:, ev] += S[:, ev] > t
                a_hits[p] += A > t
        for r, (family, severity) in enumerate(conds):
            row = dict(category=category, detector=name, family=family, severity=int(severity), regime=regime_of(family))
            for p, tag in (("clean", ""), ("aug", "_aug")):
                row[f"fpr_testcal{tag}"] = float(hits[p][r].sum() / trials[r].sum())
                row[f"tpr_testcal{tag}"] = float(np.nanmean(a_hits[p][r] / n_splits)) if A.shape[1] else np.nan
            detail.append(row)
        regimes = np.array([regime_of(f) for f, _ in conds])
        for regime in np.unique(regimes):
            m = regimes == regime
            for p in ("clean", "aug"):
                for j in range(n_img):
                    image_rows.append((name, regime, category, int(images[j]), 0, hits[p][m, j].sum(), trials[m, j].sum(), f"test-split-{p}"))
                for j in range(A.shape[1]):
                    image_rows.append((name, regime, category, int(anomaly.columns[j]), 1, a_hits[p][m, j].sum(), n_splits * m.sum(), f"test-split-{p}"))
    image_level = pd.DataFrame(image_rows, columns=["detector", "regime", "category", "index", "label", "hits", "trials", "calibration"])
    return pd.DataFrame(detail), image_level


def _hypothesis(hid, statement, a, b, alternative, primary, unit="category"):
    if a is None or b is None:
        return dict(id=hid, statement=statement, primary=primary, unit=unit, n=0, n_nonzero=0, mean_difference=np.nan,
                    ci_low=np.nan, ci_high=np.nan, p=np.nan, min_attainable_p=np.nan)
    diff = (a - b).dropna()
    nonzero = int((diff != 0).sum())
    low, high = bootstrap_mean_ci(diff.values, CONFIG["bootstrap"], seed=stable_seed(hid))
    return dict(id=hid, statement=statement, primary=primary, unit=unit, n=int(len(diff)), n_nonzero=nonzero,
                mean_difference=float(diff.mean()) if len(diff) else np.nan, ci_low=low, ci_high=high,
                p=paired_wilcoxon(diff.values, alternative), min_attainable_p=0.5 ** nonzero if nonzero else np.nan)


def _col(table, name):
    return table[name] if name in table else None


def colour_hypothesis(records, hid, statement, a, b, primary):
    clean = records[(records.split == "test") & (records.family == "clean")]
    if a not in clean.detector.values or b not in clean.detector.values:
        return _hypothesis(hid, statement, None, None, "greater", primary, unit="colour-defect image")
    margin, rank = {}, {}
    for name in (a, b):
        part = clean[clean.detector == name]
        m, r = {}, {}
        for category, g in part.groupby("category"):
            normal = g[g.label == 0].score.values
            centre = max(float(np.median(normal)), 1e-12)
            for row in g[(g.label == 1) & g.defect.str.contains("color")].itertuples():
                m[(category, row.index)] = float(np.log(max(row.score, 1e-12) / centre))
                r[(category, row.index)] = ((normal < row.score).sum() + 0.5 * (normal == row.score).sum()) / len(normal)
        margin[name], rank[name] = pd.Series(m, dtype=float), pd.Series(r, dtype=float)
    result = _hypothesis(hid, statement, margin[a], margin[b], "greater", primary, unit="category")
    diff = (margin[a] - margin[b]).dropna()
    if len(diff):
        cats = diff.index.get_level_values(0)
        groups = {c: diff[cats == c].values for c in np.unique(cats)}
        result["images"] = int(len(diff))
        result["p_image_level"] = result["p"]
        means = np.array([g.mean() for g in groups.values()])
        signs = np.array(np.meshgrid(*[[-1.0, 1.0]] * len(means))).reshape(len(means), -1).T
        flipped = (signs * np.abs(means)).mean(1)
        result["p"] = float((flipped >= means.mean() - 1e-12).mean())
        result["n"] = len(groups)
        result["n_nonzero"] = int((means != 0).sum())
        result["min_attainable_p"] = 0.5 ** result["n_nonzero"] if result["n_nonzero"] else np.nan
        if len(groups) >= 2:
            rng = np.random.default_rng(stable_seed(hid, "cluster"))
            boots = []
            for _ in range(min(CONFIG["bootstrap"], 2000)):
                pick = rng.choice(list(groups), len(groups), replace=True)
                boots.append(np.concatenate([groups[c][rng.integers(0, len(groups[c]), len(groups[c]))] for c in pick]).mean())
            result["ci_low"], result["ci_high"] = float(np.quantile(boots, 0.025)), float(np.quantile(boots, 0.975))
        result["categories"] = len(groups)
        result["rank_ceiling_share"] = float(((rank[a] == 1.0) & (rank[b] == 1.0)).mean())
    return result


def test_hypotheses(metrics, records, testcal_detail):
    held = per_category(metrics, "img_auroc", regime="heldout")
    clean = per_category(metrics, "img_auroc", family="clean")
    persp = per_category(metrics, "img_auroc", family="perspective")
    drop = clean - persp if len(persp) else pd.DataFrame()
    fpr = {}
    part = metrics[(metrics.regime == "heldout") & (metrics.detector == "patchcore_wrn")]
    if len(part):
        g = part.groupby("category")
        fpr = {p: g[f"fa_{p}"].sum() / g.n_normal.sum() for p in ("clean", "aug")}
    tc = {}
    if len(testcal_detail):
        w = testcal_detail[testcal_detail.detector == "patchcore_wrn"]
        if len(w):
            tc["held"] = w[w.regime == "heldout"].groupby("category").fpr_testcal.mean()
            tc["held_aug"] = w[w.regime == "heldout"].groupby("category").fpr_testcal_aug.mean()
            tc["clean"] = w[w.regime == "clean"].groupby("category").fpr_testcal.mean()
    rows = [
        _hypothesis("H1", "DINOv2 PatchCore beats WRN PatchCore under held-out shifts (image AUROC)",
                    _col(held, "patchcore_dino"), _col(held, "patchcore_wrn"), "greater", True),
        _hypothesis("H2", "Seen-family memory augmentation transfers to held-out families (WRN PatchCore)",
                    _col(held, "patchcore_wrn_aug"), _col(held, "patchcore_wrn"), "greater", True),
        colour_hypothesis(records, "H3", "Memory augmentation costs sensitivity to colour defects (clean, log score margin)",
                          "patchcore_wrn", "patchcore_wrn_aug", True),
        _hypothesis("H4", "PaDiM loses more than PatchCore under perspective shift (clean minus shifted AUROC)",
                    _col(drop, "padim_wrn"), _col(drop, "patchcore_wrn"), "greater", True),
        _hypothesis("H5", "Augmented calibration lowers held-out false-alarm rate (WRN PatchCore, train calibration)",
                    fpr.get("clean"), fpr.get("aug"), "greater", True),
        _hypothesis("E1", "Memory augmentation transfers for DINOv2 PatchCore",
                    _col(held, "patchcore_dino_aug"), _col(held, "patchcore_dino"), "greater", False),
        colour_hypothesis(records, "E2", "Photometric normalisation costs sensitivity to colour defects (log score margin)",
                          "patchcore_wrn", "patchcore_wrn_photonorm", False),
        _hypothesis("E3", "Photometric normalisation helps under held-out shifts",
                    _col(held, "patchcore_wrn_photonorm"), _col(held, "patchcore_wrn"), "greater", False),
        _hypothesis("E4", "Held-out shift raises false alarms even with exchangeable (test-split) calibration (WRN PatchCore)",
                    tc.get("held"), tc.get("clean"), "greater", False),
        _hypothesis("E5", "H5 repeated with test-split calibration, free of the train/test mismatch",
                    tc.get("held"), tc.get("held_aug"), "greater", False),
    ]
    table = pd.DataFrame(rows)
    for column in ("categories", "images", "p_image_level", "rank_ceiling_share"):
        if column not in table:
            table[column] = np.nan
    primary = table.primary.values.astype(bool)
    n_primary = int(primary.sum())
    table["holm_p"] = np.nan
    table.loc[primary, "holm_p"] = holm(table.loc[primary, "p"].values)

    def verdict(row):
        if row.n == 0:
            return "not testable"
        if row.n_nonzero == 0:
            return "no difference observed"
        floor = row.min_attainable_p * (n_primary if row.primary else 1)
        p = row.holm_p if row.primary else row.p
        if p < 0.05:
            return "supported"
        if floor >= 0.05:
            return f"underpowered: n={row.n_nonzero} cannot reach 0.05"
        return "not supported at this power" if row.mean_difference > 0 else "direction opposite"

    table["verdict"] = table.apply(verdict, axis=1)
    return table


def ndr_analysis(metrics):
    base = metrics[metrics.family == "clean"][["category", "detector", "img_auroc"]].rename(columns={"img_auroc": "clean_auroc"})
    d = metrics[metrics.family != "clean"].merge(base, on=["category", "detector"])
    d["auroc_drop"] = d.clean_auroc - d.img_auroc
    rows = []
    for (category, name), group in d.groupby(["category", "detector"]):
        group = group.dropna(subset=["ndr", "auroc_drop"])
        if len(group) >= 5 and group.ndr.nunique() > 1 and group.auroc_drop.nunique() > 1:
            rows.append(dict(category=category, detector=name, rho=spearmanr(group.ndr, group.auroc_drop).correlation))
    rhos = pd.DataFrame(rows).dropna() if rows else pd.DataFrame(columns=["category", "detector", "rho"])
    per_category = rhos.groupby("category").rho.median() if len(rhos) else pd.Series(dtype=float)
    positive = int((per_category > 0).sum())
    summary = dict(conditions_per_correlation=int(d.groupby(["category", "detector"]).size().median()) if len(d) else 0,
                   categories=int(len(per_category)),
                   median_spearman=float(per_category.median()) if len(per_category) else np.nan,
                   positive_categories=positive,
                   sign_test_p=float(binomtest(positive, len(per_category), 0.5, alternative="greater").pvalue) if len(per_category) else np.nan,
                   by_detector={k: round(float(v), 3) for k, v in rhos.groupby("detector").rho.median().items()} if len(rhos) else {},
                   note="needs at least 5 shifted conditions per detector and category" if not len(rhos) else "")
    return d, summary


def per_family_table(metrics):
    by_cat = metrics.groupby(["detector", "family", "category"])[["img_auroc", "img_auroc_valid", "conf_auroc"]].mean()
    return by_cat.groupby(["detector", "family"]).mean().round(4)


def calibration_tradeoff_table(alarms):
    part = alarms[(alarms.regime == "heldout") & alarms.calibration.isin(["test-split-clean", "test-split-aug"])]
    wide = part.pivot_table(index="detector", columns="calibration", values=["fpr", "tpr"])
    wide.columns = [f"{v}_{c.replace('test-split-', '')}" for v, c in wide.columns]
    return wide.round(4)


def _stratified_permutation_p(values, labels, strata, n_perm, seed):
    def stat(lab):
        total, weight = 0.0, 0
        for s in np.unique(strata):
            m = strata == s
            v, l = values[m], lab[m]
            if l.sum() and (~l).sum():
                total += (v[l].mean() - v[~l].mean()) * l.sum()
                weight += l.sum()
        return total / weight if weight else np.nan
    observed = stat(labels)
    rng = np.random.default_rng(seed)
    count = 0
    for _ in range(n_perm):
        shuffled = labels.copy()
        for s in np.unique(strata):
            m = np.where(strata == s)[0]
            shuffled[m] = labels[m][rng.permutation(len(m))]
        count += stat(shuffled) >= observed
    return observed, (count + 1) / (n_perm + 1)


def colour_specific_control(records, a, b):
    clean = records[(records.split == "test") & (records.family == "clean")]
    if a not in clean.detector.values or b not in clean.detector.values:
        return dict(test="colour-specific cost", n=0)
    cats = clean[(clean.label == 1) & clean.defect.str.contains("color")].category.unique()
    rows = []
    for name in (a, b):
        part = clean[(clean.detector == name) & clean.category.isin(cats)]
        for category, g in part.groupby("category"):
            centre = max(float(np.median(g[g.label == 0].score)), 1e-12)
            for row in g[g.label == 1].itertuples():
                rows.append((name, category, row.index, "color" in row.defect, float(np.log(max(row.score, 1e-12) / centre))))
    frame = pd.DataFrame(rows, columns=["detector", "category", "index", "colour", "margin"])
    wide = frame.pivot_table(index=["category", "index", "colour"], columns="detector", values="margin").reset_index().dropna()
    loss = (wide[a] - wide[b]).values
    labels = wide.colour.values.astype(bool)
    strata = wide.category.values
    observed, p = _stratified_permutation_p(loss, labels, strata, 5000, stable_seed("colour-did"))
    return dict(test="colour-specific cost: margin loss on colour defects minus margin loss on other defects",
                categories=int(len(cats)), colour_images=int(labels.sum()), other_images=int((~labels).sum()),
                mean_loss_colour=round(float(loss[labels].mean()), 4), mean_loss_other=round(float(loss[~labels].mean()), 4),
                difference=round(float(observed), 4), permutation_p=round(float(p), 5))


def geometric_border_control(metrics):
    out = {}
    for family in ("rotation", "perspective"):
        raw = per_category(metrics, "img_auroc", family=family)
        valid = per_category(metrics, "img_auroc_valid", family=family)
        if "patchcore_wrn" in raw and "patchcore_wrn_aug" in raw:
            for tag, table in (("raw", raw), ("valid", valid)):
                diff = (table["patchcore_wrn_aug"] - table["patchcore_wrn"]).dropna()
                out[f"{family}_aug_gain_{tag}"] = round(float(diff.mean()), 4)
                out[f"{family}_aug_gain_{tag}_p"] = paired_wilcoxon(diff.values, "greater")
    held_valid = per_category(metrics, "img_auroc_valid", regime="heldout")
    if "patchcore_wrn" in held_valid and "patchcore_wrn_aug" in held_valid:
        diff = (held_valid["patchcore_wrn_aug"] - held_valid["patchcore_wrn"]).dropna()
        out["H2_valid_gain"] = round(float(diff.mean()), 4)
        out["H2_valid_p"] = paired_wilcoxon(diff.values, "greater")
        out["H2_valid_note"] = "H2 ranked 4th of 5 primary tests, so under Holm it needs p < 0.025"
    clean = per_category(metrics, "img_auroc", family="clean")
    persp_valid = per_category(metrics, "img_auroc_valid", family="perspective")
    if "padim_wrn" in persp_valid and "patchcore_wrn" in persp_valid:
        drop = clean - persp_valid
        diff = (drop["padim_wrn"] - drop["patchcore_wrn"]).dropna()
        out["H4_valid_padim_minus_patchcore_drop"] = round(float(diff.mean()), 4)
        out["H4_valid_p"] = paired_wilcoxon(diff.values, "greater")
    nuis = metrics[metrics.group == "geometric"].groupby("detector")[["nuis_auroc", "nuis_auroc_valid"]].mean().round(4)
    out["nuisance_auroc_raw_vs_valid"] = nuis.to_dict(orient="index")
    return out


def margin_sensitivity(metrics):
    rows = []
    o, c, size = crop_offset(), CONFIG["crop_size"], CONFIG["image_size"]
    top = max(CONFIG["severities"])
    for k, m in enumerate(CONFIG["valid_margins"]):
        tag = "" if k == 0 else f"_m{m}"
        row = {"margin_px": m}
        for family in ("rotation", "perspective"):
            kept = [valid_region(family, top, ("margin", family, i), size)[m][o:o + c, o:o + c].mean() for i in range(100)]
            row[f"{family}_area_kept_s{top}"] = round(float(np.mean(kept)), 3)
        persp = per_category(metrics, f"img_auroc_valid{tag}", family="perspective")
        held = per_category(metrics, f"img_auroc_valid{tag}", regime="heldout")
        clean = per_category(metrics, "img_auroc", family="clean")
        if "patchcore_wrn" in persp:
            row["patchcore_wrn_perspective_auroc"] = round(float(persp["patchcore_wrn"].mean()), 4)
        if "patchcore_wrn" in held and "patchcore_wrn_aug" in held:
            diff = (held["patchcore_wrn_aug"] - held["patchcore_wrn"]).dropna()
            row["H2_gain"], row["H2_p"] = round(float(diff.mean()), 4), paired_wilcoxon(diff.values, "greater")
            diff = (persp["patchcore_wrn_aug"] - persp["patchcore_wrn"]).dropna()
            row["perspective_aug_gain"], row["perspective_aug_p"] = round(float(diff.mean()), 4), paired_wilcoxon(diff.values, "greater")
        if "padim_wrn" in persp and "patchcore_wrn" in persp:
            drop = clean - persp
            diff = (drop["padim_wrn"] - drop["patchcore_wrn"]).dropna()
            row["H4_difference"], row["H4_p"] = round(float(diff.mean()), 4), paired_wilcoxon(diff.values, "greater")
        geo = metrics[(metrics.group == "geometric") & (metrics.detector == "patchcore_wrn")]
        if len(geo) and f"pix_auroc{tag}" in geo:
            row["patchcore_wrn_geometric_pix_auroc"] = round(float(geo.groupby("category")[f"pix_auroc{tag}"].mean().mean()), 4)
            row["patchcore_wrn_geometric_aupro"] = round(float(geo.groupby("category")[f"aupro{tag}"].mean().mean()), 4)
        rows.append(row)
    return pd.DataFrame(rows).set_index("margin_px")


def ndr_fixed_severity(ndr_frame):
    rows = []
    for (category, name, severity), g in ndr_frame.groupby(["category", "detector", "severity"]):
        g = g.dropna(subset=["ndr", "auroc_drop"])
        if len(g) >= 5 and g.ndr.nunique() > 1 and g.auroc_drop.nunique() > 1:
            rows.append(dict(category=category, detector=name, severity=severity, rho=spearmanr(g.ndr, g.auroc_drop).correlation))
    rhos = pd.DataFrame(rows).dropna()
    if rhos.empty:
        return dict(test="NDR across families at fixed severity", categories=0)
    per_category = rhos.groupby("category").rho.median()
    positive = int((per_category > 0).sum())
    return dict(test="NDR across families at fixed severity", families_per_correlation=int(ndr_frame.groupby(["category", "detector", "severity"]).size().median()),
                categories=int(len(per_category)), median_spearman=round(float(per_category.median()), 4),
                positive_categories=positive,
                sign_test_p=float(binomtest(positive, len(per_category), 0.5, alternative="greater").pvalue),
                by_severity={int(k): round(float(v), 3) for k, v in rhos.groupby("severity").rho.median().items()})


FOLLOWUP_PREREGISTRATION = """Follow-up experiment, pre-registered 2026-11-01 before any local-contrast score was computed.
Motivation (from run v2): the worst false alarms light up almost the whole map (median 99.7% of pixels above
threshold), and nuisances inflate normal scores about twice as much as defect scores in log terms. A nuisance
lifts every patch; a defect lifts a few. score_local = max - median patch score, both over real content.
Baseline for every test: score_valid (max over real content). Detector: PatchCore WRN. Unit: category unless stated.
L1  score_local raises held-out image AUROC.
L2  With test-split calibration (each score calibrated on its own), score_local lowers held-out false alarms.
L3  score_local costs less than 0.01 clean image AUROC (non-inferiority, margin 0.01).
L4  Any clean loss comes from large defects: per-image rank change correlates negatively with defect area.
Holm across L1 to L4. All other detectors are reported descriptively."""


def _image_ranks(clean, column):
    out = {}
    for category, g in clean.groupby("category"):
        normal = g[g.label == 0][column].values
        for row in g[g.label == 1].itertuples():
            v = getattr(row, column)
            out[(category, row.index)] = ((normal < v).sum() + 0.5 * (normal == v).sum()) / len(normal)
    return pd.Series(out, dtype=float)


def followup_tests(metrics, records, detector="patchcore_wrn"):
    rows = []
    held_v = per_category(metrics, "img_auroc_valid", regime="heldout")
    held_l = per_category(metrics, "img_auroc_local", regime="heldout")
    rows.append(_hypothesis("L1", "Local-contrast score raises held-out image AUROC",
                            _col(held_l, detector), _col(held_v, detector), "greater", True))
    tc_v, _ = test_split_calibration(records[records.detector == detector], column="score_valid")
    tc_l, _ = test_split_calibration(records[records.detector == detector], column="score_local")
    if len(tc_v) and len(tc_l):
        fv = tc_v[tc_v.regime == "heldout"].groupby("category").fpr_testcal.mean()
        fl = tc_l[tc_l.regime == "heldout"].groupby("category").fpr_testcal.mean()
        rows.append(_hypothesis("L2", "Local-contrast score lowers held-out false alarms (test-split calibration)",
                                fv, fl, "greater", True))
        tv = tc_v[tc_v.regime == "heldout"].groupby("category").tpr_testcal.mean()
        tl = tc_l[tc_l.regime == "heldout"].groupby("category").tpr_testcal.mean()
        tradeoff = dict(heldout_fpr_valid=round(float(fv.mean()), 4), heldout_fpr_local=round(float(fl.mean()), 4),
                        heldout_tpr_valid=round(float(tv.mean()), 4), heldout_tpr_local=round(float(tl.mean()), 4),
                        clean_fpr_local=round(float(tc_l[tc_l.regime == "clean"].fpr_testcal.mean()), 4))
    else:
        rows.append(_hypothesis("L2", "Local-contrast score lowers held-out false alarms", None, None, "greater", True))
        tradeoff = {}
    clean_v = per_category(metrics, "img_auroc_valid", family="clean")
    clean_l = per_category(metrics, "img_auroc_local", family="clean")
    l3 = _hypothesis("L3", "Local-contrast score costs less than 0.01 clean AUROC (non-inferiority)",
                     _col(clean_l, detector) + 0.01 if detector in clean_l else None, _col(clean_v, detector), "greater", True)
    l3["mean_difference"] = l3["mean_difference"] - 0.01 if l3["n"] else np.nan
    for k in ("ci_low", "ci_high"):
        if l3["n"] and not np.isnan(l3[k]):
            l3[k] -= 0.01
    rows.append(l3)
    clean = records[(records.split == "test") & (records.family == "clean") & (records.detector == detector)]
    if len(clean):
        change = _image_ranks(clean, "score_local") - _image_ranks(clean, "score_valid")
        area = clean[clean.label == 1].set_index(["category", "index"]).defect_area
        frame = pd.DataFrame({"change": change, "area": area.reindex(change.index)}).dropna()
        rhos = []
        for category, g in frame.groupby(level=0):
            if len(g) >= 10 and g.change.nunique() > 1 and g.area.nunique() > 1:
                rhos.append(spearmanr(g.area, g.change).correlation)
        rhos = np.array([r for r in rhos if not np.isnan(r)])
        negative = int((rhos < 0).sum())
        p = float(binomtest(negative, len(rhos), 0.5, alternative="greater").pvalue) if len(rhos) else np.nan
        rows.append(dict(id="L4", statement="Clean rank loss concentrates in large defects (per-category Spearman < 0)",
                         primary=True, unit="category", n=int(len(rhos)), n_nonzero=int(len(rhos)),
                         mean_difference=float(np.median(rhos)) if len(rhos) else np.nan, ci_low=np.nan, ci_high=np.nan,
                         p=p, min_attainable_p=0.5 ** len(rhos) if len(rhos) else np.nan))
    table = pd.DataFrame(rows)
    table["holm_p"] = holm(table.p.values)

    def verdict(row):
        if row.n == 0:
            return "not testable"
        if np.isnan(row.p):
            return "no difference observed"
        if row.holm_p < 0.05:
            return "supported"
        if row.min_attainable_p * len(table) >= 0.05:
            return f"underpowered: n={row.n}"
        return "not supported"
    table["verdict"] = table.apply(verdict, axis=1)
    descriptive = metrics.groupby(["detector", "regime", "category"])[["img_auroc_valid", "img_auroc_local", "conf_auroc", "conf_auroc_local"]].mean()
    descriptive = descriptive.groupby(["detector", "regime"]).mean().round(4)
    return table, tradeoff, descriptive


TEXTURE_CATEGORIES = ["carpet", "grid", "leather", "tile", "wood"]


def exploratory_local(metrics, records):
    out = {}
    test = records[records.split == "test"]
    held = CONFIG["heldout_families"]
    floors = {}
    for name, part in test.groupby("detector"):
        row = {}
        for column in ("score_valid", "score_local"):
            fpr = []
            for category, g in part.groupby("category"):
                top = g[(g.family == "clean") & (g.label == 0)][column].max()
                h = g[g.family.isin(held) & (g.label == 0)]
                fpr.append(float((h[column] > top).mean()))
            row[column] = round(float(np.mean(fpr)), 4)
        floors[name] = row
    out["lowest_heldout_false_alarm_reachable_with_clean_calibration"] = floors
    m = metrics.assign(kind=np.where(metrics.category.isin(TEXTURE_CATEGORIES), "texture", "object"))
    cols = ["img_auroc_valid", "img_auroc_local", "conf_auroc", "conf_auroc_local"]
    out["texture_vs_object"] = (m[m.regime.isin(["clean", "heldout"])].groupby(["detector", "kind", "regime", "category"])[cols]
                                .mean().groupby(["detector", "kind", "regime"]).mean().round(4))
    w = test[test.detector == "patchcore_wrn"]
    rows = []
    for category, g in w.groupby("category"):
        row = {"category": category}
        for column in ("score_valid", "score_local"):
            thr = np.quantile(g[(g.family == "clean") & (g.label == 0)][column], 1 - CONFIG["alpha"])
            h = g[g.family.isin(held)]
            row[f"fpr_{column}"] = round(float((h[h.label == 0][column] > thr).mean()), 4)
            row[f"tpr_{column}"] = round(float((h[h.label == 1][column] > thr).mean()), 4)
        rows.append(row)
    out["per_category_heldout_patchcore_wrn"] = pd.DataFrame(rows).set_index("category").sort_values("fpr_score_local") if rows else pd.DataFrame()
    return out


DETECTOR_STYLE = {"patchcore_wrn": ("#1f4e79", "-"), "patchcore_wrn_aug": ("#1f4e79", "--"),
                  "patchcore_wrn_photonorm": ("#1f4e79", ":"), "patchcore_dino": ("#b2502a", "-"),
                  "patchcore_dino_aug": ("#b2502a", "--"), "padim_wrn": ("#4f7d3a", "-"), "padim_wrn_aug": ("#4f7d3a", "--")}


def _save(fig, name):
    folder = Path(CONFIG["output_dir"]) / "figures"
    folder.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(folder / name, dpi=160)
    plt.close(fig)


def figure_robustness_curves(metrics):
    families = [f for f in CONFIG["seen_families"] + CONFIG["heldout_families"] if f in metrics.family.values]
    cols = 4
    rows_n = int(math.ceil(len(families) / cols))
    fig, axes = plt.subplots(rows_n, cols, figsize=(3.4 * cols, 2.8 * rows_n), squeeze=False)
    clean = metrics[metrics.family == "clean"]
    for ax, family in zip(axes.ravel(), families):
        part = metrics[metrics.family == family]
        cats = part.category.unique()
        for name in part.detector.unique():
            color, style = DETECTOR_STYLE.get(name, ("grey", "-"))
            c0 = clean[(clean.detector == name) & clean.category.isin(cats)].img_auroc.mean()
            curve = part[part.detector == name].groupby("severity").img_auroc.mean()
            ax.plot([0] + list(curve.index), [c0] + list(curve.values), color=color, linestyle=style, marker="o", ms=3, label=name)
        ax.set_title(f"{family} ({regime_of(family)})", fontsize=9)
        ax.set_xlabel("severity")
        ax.set_ylabel("image AUROC")
    for ax in axes.ravel()[len(families):]:
        ax.axis("off")
    axes.ravel()[0].legend(fontsize=6)
    _save(fig, "fig1_robustness_curves.png")


def figure_confusion(metrics):
    top = max(CONFIG["severities"])
    part = metrics[(metrics.family != "clean") & (metrics.severity == top)]
    table = part.groupby(["detector", "family"]).conf_auroc.mean().unstack("family")
    fig, ax = plt.subplots(figsize=(1.1 * table.shape[1] + 2.5, 0.45 * table.shape[0] + 1.5))
    im = ax.imshow(table.values, cmap="RdBu", vmin=0.0, vmax=1.0, aspect="auto")
    ax.set_xticks(range(table.shape[1]))
    ax.set_xticklabels(table.columns, rotation=40, ha="right", fontsize=8)
    ax.set_yticks(range(table.shape[0]))
    ax.set_yticklabels(table.index, fontsize=8)
    for i in range(table.shape[0]):
        for j in range(table.shape[1]):
            v = table.values[i, j]
            ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=7, color="white" if abs(v - 0.5) > 0.3 else "black")
    ax.set_title(f"AUROC, shifted normal vs clean defect (severity {top}); below 0.5 = normals outscore defects", fontsize=8)
    fig.colorbar(im, ax=ax)
    _save(fig, "fig2_nuisance_vs_defect.png")


def figure_invariance_sensitivity(metrics):
    families = [f for f in CONFIG["seen_families"] + CONFIG["heldout_families"] if f in metrics.family.values]
    pairs = [(b, a) for b, a in [("patchcore_wrn", "patchcore_wrn_aug"), ("patchcore_dino", "patchcore_dino_aug"),
                                 ("padim_wrn", "padim_wrn_aug")] if a in metrics.detector.values]
    fig, ax = plt.subplots(figsize=(1.0 * len(families) + 3, 3.6))
    width = 0.8 / max(len(pairs), 1)
    for k, (base, aug) in enumerate(pairs):
        gains = []
        for f in families:
            t = per_category(metrics, "img_auroc_valid", family=f)
            gains.append(float((t[aug] - t[base]).mean()) if base in t and aug in t else np.nan)
        ax.bar(np.arange(len(families)) + (k - (len(pairs) - 1) / 2) * width, gains, width,
               color=DETECTOR_STYLE[base][0], label=f"{aug} minus {base}")
    seen_n = sum(f in CONFIG["seen_families"] for f in families)
    ax.axvspan(-0.5, seen_n - 0.5, color="grey", alpha=0.12)
    ax.text(seen_n / 2 - 0.5, ax.get_ylim()[1] * 0.92, "seen", ha="center", fontsize=8)
    ax.text(seen_n + (len(families) - seen_n) / 2 - 0.5, ax.get_ylim()[1] * 0.92, "held out", ha="center", fontsize=8)
    ax.axhline(0, color="black", lw=0.8)
    ax.set_xticks(range(len(families)))
    ax.set_xticklabels(families, rotation=30, ha="right", fontsize=8)
    ax.set_ylabel("AUROC gain (real content)")
    ax.legend(fontsize=6)
    _save(fig, "fig3_augmentation_transfer.png")


def figure_false_alarms(fa_table):
    regimes = [r for r in ("clean", "heldout") if r in fa_table.regime.values]
    policies = [p for p in ("train-clean", "train-aug", "test-split-clean", "test-split-aug") if p in fa_table.calibration.values]
    colours = {"train-clean": "#7ba0c4", "train-aug": "#1f4e79", "test-split-clean": "#e0a080", "test-split-aug": "#b2502a"}
    detectors = sorted(fa_table.detector.unique())
    fig, axes = plt.subplots(2, len(regimes), figsize=(6.5 * len(regimes), 6.5), squeeze=False, sharex=True)
    width = 0.8 / len(policies)
    for j, regime in enumerate(regimes):
        for k, policy in enumerate(policies):
            part = fa_table[(fa_table.regime == regime) & (fa_table.calibration == policy)].set_index("detector").reindex(detectors)
            x = np.arange(len(detectors)) + (k - (len(policies) - 1) / 2) * width
            y = part.fpr.values.astype(float)
            err = np.vstack([np.clip(y - part.fpr_low.values.astype(float), 0, None), np.clip(part.fpr_high.values.astype(float) - y, 0, None)])
            axes[0, j].bar(x, y, width, yerr=np.nan_to_num(err), capsize=2, color=colours[policy], label=policy)
            axes[1, j].bar(x, part.tpr.values.astype(float), width, color=colours[policy])
        axes[0, j].axhline(CONFIG["alpha"], color="black", linestyle="--", lw=0.8)
        axes[0, j].set_title(f"{regime}: false-alarm rate (95% CI)", fontsize=9)
        axes[1, j].set_title(f"{regime}: detection rate", fontsize=9)
        axes[1, j].set_xticks(range(len(detectors)))
        axes[1, j].set_xticklabels(detectors, rotation=35, ha="right", fontsize=7)
        axes[0, j].set_ylim(0, 1)
        axes[1, j].set_ylim(0, 1)
    axes[0, 0].legend(fontsize=7)
    _save(fig, "fig4_false_alarms.png")


def figure_ndr(ndr_frame):
    rows = []
    for (category, name), g in ndr_frame.groupby(["category", "detector"]):
        g = g.dropna(subset=["ndr", "auroc_drop"])
        if len(g) >= 5 and g.ndr.nunique() > 1 and g.auroc_drop.nunique() > 1:
            rows.append(dict(category=category, analysis="all conditions", rho=spearmanr(g.ndr, g.auroc_drop).correlation))
        for severity, h in g.groupby("severity"):
            if len(h) >= 5 and h.ndr.nunique() > 1 and h.auroc_drop.nunique() > 1:
                rows.append(dict(category=category, analysis="fixed severity", rho=spearmanr(h.ndr, h.auroc_drop).correlation))
    frame = pd.DataFrame(rows).dropna()
    if frame.empty:
        return
    per = frame.groupby(["category", "analysis"]).rho.median().unstack("analysis")
    per = per.sort_values(per.columns[0])
    fig, ax = plt.subplots(figsize=(5.5, 0.28 * len(per) + 1.5))
    for k, column in enumerate(per.columns):
        ax.scatter(per[column], np.arange(len(per)) + (k - 0.5) * 0.2, label=column, s=20, color=["#1f4e79", "#b2502a"][k % 2])
    ax.axvline(0, color="black", lw=0.8)
    ax.set_yticks(range(len(per)))
    ax.set_yticklabels(per.index, fontsize=7)
    ax.set_xlim(-1, 1)
    ax.set_xlabel("Spearman: NDR (normals only) vs AUROC drop, median over detectors")
    ax.legend(fontsize=7)
    _save(fig, "fig5_ndr_predicts_drop.png")


def figure_local_contrast(metrics, detector="patchcore_wrn"):
    families = [f for f in CONFIG["seen_families"] + CONFIG["heldout_families"] if f in metrics.family.values]
    if "img_auroc_local" not in metrics:
        return
    d = metrics[metrics.detector == detector]
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
    for ax, (column, title) in zip(axes, [(("img_auroc_valid", "img_auroc_local"), "image AUROC"),
                                          (("conf_auroc", "conf_auroc_local"), "shifted normal vs clean defect AUROC")]):
        base = [d[d.family == f].groupby("category")[column[0]].mean().mean() for f in ["clean"] + families]
        local = [d[d.family == f].groupby("category")[column[1]].mean().mean() for f in ["clean"] + families]
        x = np.arange(len(base))
        ax.bar(x - 0.2, base, 0.4, color="#7ba0c4", label="max score")
        ax.bar(x + 0.2, local, 0.4, color="#1f4e79", label="max minus median")
        ax.set_xticks(x)
        ax.set_xticklabels(["clean"] + families, rotation=30, ha="right", fontsize=8)
        ax.set_ylim(0, 1)
        ax.axhline(0.5, color="grey", lw=0.6, linestyle=":")
        ax.set_title(f"{detector}: {title}", fontsize=9)
    axes[0].legend(fontsize=7)
    _save(fig, "fig6_local_contrast.png")


def figure_gallery(category):
    folder = Path(CONFIG["output_dir"]) / "categories" / category
    if not (folder / "gallery.json").exists():
        return
    meta = json.load(open(folder / "gallery.json"))
    arrays = np.load(folder / "gallery.npz")
    threshold = json.load(open(folder / "thresholds.json"))["thresholds"].get(CONFIG["gallery_detector"], {}).get("clean", 1.0)
    k = CONFIG["gallery_k"]
    fig, axes = plt.subplots(2, k, figsize=(1.9 * k, 4.2), squeeze=False)
    for r, pool in enumerate(["false_alarms", "misses"]):
        for j in range(k):
            ax = axes[r, j]
            ax.axis("off")
            if j >= len(meta[pool]):
                continue
            item = meta[pool][j]
            ax.imshow(arrays[f"{pool}_{j}_image"])
            scale = threshold if np.isfinite(threshold) and threshold > 0 else arrays[f"{pool}_{j}_map"].max()
            ax.imshow(arrays[f"{pool}_{j}_map"] / scale, cmap="jet", alpha=0.45, vmin=0, vmax=1.5)
            ax.set_title(f"{item['family']} s{item['severity']}\n{item['defect']} {item['score']:.2f}", fontsize=6)
    axes[0, 0].text(-0.15, 0.5, "false alarms", transform=axes[0, 0].transAxes, rotation=90, va="center", fontsize=8)
    axes[1, 0].text(-0.15, 0.5, "missed defects", transform=axes[1, 0].transAxes, rotation=90, va="center", fontsize=8)
    _save(fig, f"gallery_{category}.png")


def write_report(tables):
    lines = [f"Run finished {time.strftime('%Y-%m-%d %H:%M')}", f"alpha = {CONFIG['alpha']}",
             f"seen families: {CONFIG['seen_families']}", f"held-out families: {CONFIG['heldout_families']}", ""]
    for title, frame in tables.items():
        if isinstance(frame, str):
            body = frame
        elif hasattr(frame, "to_string"):
            body = frame.to_string()
        else:
            body = json.dumps(frame, indent=1, default=str)
        lines += [title, body, ""]
    text = "\n".join(lines)
    with open(Path(CONFIG["output_dir"]) / "report.txt", "w") as handle:
        handle.write(text)
    return text

In [10]:
def _onnx_export(module, dummy, path):
    kwargs = dict(input_names=["image"], output_names=["features"], opset_version=17,
                  dynamic_axes={"image": {0: "batch"}, "features": {0: "batch"}})
    if "dynamo" in inspect.signature(torch.onnx.export).parameters:
        kwargs["dynamo"] = False
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=DeprecationWarning)
        warnings.simplefilter("ignore", category=FutureWarning)
        torch.onnx.export(module, dummy, str(path), **kwargs)


def _prepare_single(path):
    image = read_rgb(path, CONFIG["image_size"])[None]
    o, c = crop_offset(), CONFIG["crop_size"]
    x = image[:, o:o + c, o:o + c, :].astype(np.float32) / 255.0
    x = (x - np.array([0.485, 0.456, 0.406], np.float32)) / np.array([0.229, 0.224, 0.225], np.float32)
    return np.ascontiguousarray(x.transpose(0, 3, 1, 2))


def _numpy_score(features, bank):
    q = features[0].reshape(features.shape[1], -1).T.astype(np.float64)
    b = bank.astype(np.float64)
    d2 = (q * q).sum(1, keepdims=True) + (b * b).sum(1)[None] - 2 * q @ b.T
    return float(np.sqrt(np.clip(d2.min(1), 0, None)).max())


def export_deployment(models):
    import onnxruntime as ort
    folder = Path(CONFIG["output_dir"]) / "deploy"
    folder.mkdir(parents=True, exist_ok=True)
    trunk = copy.deepcopy(models["wrn"]).cpu().eval()
    net = PatchCoreWRN(trunk).eval()
    dummy = torch.randn(1, 3, CONFIG["crop_size"], CONFIG["crop_size"])
    onnx_path = folder / "patchcore_wrn.onnx"
    _onnx_export(net, dummy, onnx_path)
    threads = torch.get_num_threads()
    options = ort.SessionOptions()
    options.intra_op_num_threads = threads
    options.inter_op_num_threads = 1
    session = ort.InferenceSession(str(onnx_path), sess_options=options, providers=["CPUExecutionProvider"])
    with torch.inference_mode():
        reference = net(dummy).numpy()
    produced = session.run(None, {"image": dummy.numpy()})[0]
    report = dict(onnx_max_abs_diff=float(np.abs(reference - produced).max()),
                  onnx_rel_diff=float(np.abs(reference - produced).max() / (np.abs(reference).max() + 1e-12)))

    def run_torch():
        with torch.inference_mode():
            net(dummy)

    def run_ort():
        session.run(None, {"image": dummy.numpy()})

    times = {"torch": [], "ort": []}
    for i in range(110):
        order = [("torch", run_torch), ("ort", run_ort)] if i % 2 == 0 else [("ort", run_ort), ("torch", run_torch)]
        for label, fn in order:
            t = time.perf_counter()
            fn()
            if i >= 10:
                times[label].append(1000 * (time.perf_counter() - t))
    q = lambda x: [round(float(v), 1) for v in np.percentile(x, [25, 50, 75])]
    ratio = np.array(times["torch"]) / np.array(times["ort"])
    report.update(torch_cpu_ms_q25_median_q75=q(times["torch"]), onnxruntime_cpu_ms_q25_median_q75=q(times["ort"]),
                  torch_over_ort_ratio_q25_median_q75=[round(float(v), 2) for v in np.percentile(ratio, [25, 50, 75])],
                  timed_runs=len(times["torch"]), cpu_threads=threads)

    info_path = folder / "deploy_info.json"
    if info_path.exists():
        info = json.load(open(info_path))
        bank = np.fromfile(folder / "bank.bin", dtype=np.float32).reshape(info["bank_rows"], info["dim"])
        for tag, ref in info["references"].items():
            features = session.run(None, {"image": _prepare_single(ref["image"])})[0]
            ref["onnx_score"] = _numpy_score(features, bank)
            ref["abs_diff"] = abs(ref["onnx_score"] - ref["python_score"])
            ref["decision"] = "anomalous" if ref["onnx_score"] > info["threshold"] else "normal"
        report["references"] = info["references"]
        with open(folder / "meta.txt", "w") as handle:
            handle.write(f"bank_rows {info['bank_rows']}\ndim {info['dim']}\ngrid {CONFIG['crop_size'] // 8}\n"
                         f"image_size {CONFIG['image_size']}\ncrop_size {CONFIG['crop_size']}\n"
                         f"threshold {info['threshold'] if np.isfinite(info['threshold']) else 1e30}\n"
                         f"blur_sigma {CONFIG['blur_sigma']}\n")
    write_cpp_sources(folder)
    if CONFIG["verify_cpp"] and "references" in report:
        report["cpp"] = verify_cpp(folder, report["references"], info["threshold"])
    with open(folder / "parity_report.json", "w") as handle:
        json.dump(report, handle, indent=1)
    return report


CPP_MAIN = r'''// PatchCore scoring with OpenCV + ONNX Runtime, brute-force nearest neighbour
// Usage: patchcore_infer patchcore_wrn.onnx bank.bin meta.txt image.png [overlay.png]
#include <onnxruntime_cxx_api.h>
#include <opencv2/core.hpp>
#include <opencv2/imgcodecs.hpp>
#include <opencv2/imgproc.hpp>
#include <algorithm>
#include <chrono>
#include <cmath>
#include <fstream>
#include <iostream>
#include <limits>
#include <map>
#include <string>
#include <vector>

static std::map<std::string, double> read_meta(const std::string& path) {
    std::map<std::string, double> meta;
    std::ifstream in(path);
    std::string key;
    double value;
    while (in >> key >> value) meta[key] = value;
    return meta;
}

int main(int argc, char** argv) {
    if (argc < 5) {
        std::cerr << "usage: patchcore_infer model.onnx bank.bin meta.txt image.png [overlay.png]\n";
        return 1;
    }
    auto meta = read_meta(argv[3]);
    const int rows = static_cast<int>(meta["bank_rows"]), dim = static_cast<int>(meta["dim"]);
    const int image_size = static_cast<int>(meta["image_size"]), crop = static_cast<int>(meta["crop_size"]);
    const float threshold = static_cast<float>(meta["threshold"]), sigma = static_cast<float>(meta["blur_sigma"]);

    std::vector<float> bank(static_cast<size_t>(rows) * dim);
    std::ifstream bank_file(argv[2], std::ios::binary);
    bank_file.read(reinterpret_cast<char*>(bank.data()), bank.size() * sizeof(float));
    if (!bank_file) { std::cerr << "bank.bin is shorter than meta.txt says\n"; return 1; }
    // double accumulation: |q|^2 + |b|^2 - 2 q.b cancels badly in float when the distance is small
    std::vector<double> bank_sq(rows, 0.0);
    for (int r = 0; r < rows; ++r)
        for (int k = 0; k < dim; ++k) {
            const double v = bank[static_cast<size_t>(r) * dim + k];
            bank_sq[r] += v * v;
        }

    Ort::Env env(ORT_LOGGING_LEVEL_WARNING, "patchcore");
    Ort::SessionOptions options;
    options.SetGraphOptimizationLevel(GraphOptimizationLevel::ORT_ENABLE_ALL);
    Ort::Session session(env, argv[1], options);

    cv::Mat bgr = cv::imread(argv[4], cv::IMREAD_COLOR);
    if (bgr.empty()) { std::cerr << "cannot read image\n"; return 1; }
    auto t0 = std::chrono::steady_clock::now();
    cv::Mat rgb, resized;
    cv::cvtColor(bgr, rgb, cv::COLOR_BGR2RGB);
    cv::resize(rgb, resized, cv::Size(image_size, image_size), 0, 0, cv::INTER_AREA);
    const int offset = (image_size - crop) / 2;
    cv::Mat cropped = resized(cv::Rect(offset, offset, crop, crop)).clone();
    const float mean[3] = {0.485f, 0.456f, 0.406f}, stdv[3] = {0.229f, 0.224f, 0.225f};
    std::vector<float> input(3 * crop * crop);
    for (int y = 0; y < crop; ++y)
        for (int x = 0; x < crop; ++x) {
            const cv::Vec3b px = cropped.at<cv::Vec3b>(y, x);
            for (int c = 0; c < 3; ++c) input[c * crop * crop + y * crop + x] = (px[c] / 255.f - mean[c]) / stdv[c];
        }

    Ort::MemoryInfo memory = Ort::MemoryInfo::CreateCpu(OrtArenaAllocator, OrtMemTypeDefault);
    std::vector<int64_t> shape = {1, 3, crop, crop};
    Ort::Value tensor = Ort::Value::CreateTensor<float>(memory, input.data(), input.size(), shape.data(), shape.size());
    const char* input_names[] = {"image"};
    const char* output_names[] = {"features"};
    auto t1 = std::chrono::steady_clock::now();
    auto outputs = session.Run(Ort::RunOptions{nullptr}, input_names, &tensor, 1, output_names, 1);
    auto t2 = std::chrono::steady_clock::now();
    const float* features = outputs[0].GetTensorData<float>();
    auto out_shape = outputs[0].GetTensorTypeAndShapeInfo().GetShape();
    const int channels = static_cast<int>(out_shape[1]), gh = static_cast<int>(out_shape[2]), gw = static_cast<int>(out_shape[3]);
    if (channels != dim) { std::cerr << "feature dim " << channels << " != bank dim " << dim << "\n"; return 1; }

    cv::Mat grid(gh, gw, CV_32F);
    std::vector<float> q(dim);
    float image_score = 0.f;
    for (int p = 0; p < gh * gw; ++p) {
        double q_sq = 0.0;
        for (int k = 0; k < dim; ++k) { q[k] = features[k * gh * gw + p]; q_sq += static_cast<double>(q[k]) * q[k]; }
        double best = std::numeric_limits<double>::max();
        for (int r = 0; r < rows; ++r) {
            const float* b = &bank[static_cast<size_t>(r) * dim];
            double dot = 0.0;
            for (int k = 0; k < dim; ++k) dot += static_cast<double>(q[k]) * b[k];
            best = std::min(best, q_sq + bank_sq[r] - 2.0 * dot);
        }
        const float s = static_cast<float>(std::sqrt(std::max(best, 0.0)));
        grid.at<float>(p / gw, p % gw) = s;
        image_score = std::max(image_score, s);
    }
    auto t3 = std::chrono::steady_clock::now();

    std::cout.setf(std::ios::fixed);
    std::cout.precision(6);
    std::cout << "score " << image_score << "\nthreshold " << threshold << "\ndecision "
              << (image_score > threshold ? "anomalous" : "normal") << "\n";
    auto ms = [](auto a, auto b) { return std::chrono::duration<double, std::milli>(b - a).count(); };
    std::cout.precision(1);
    std::cout << "preprocess_ms " << ms(t0, t1) << "\nnetwork_ms " << ms(t1, t2) << "\nknn_ms " << ms(t2, t3) << "\n";

    if (argc > 5) {
        cv::Mat up, blurred, scaled, colour, base, overlay;
        cv::resize(grid, up, cv::Size(crop, crop), 0, 0, cv::INTER_LINEAR);
        const int radius = static_cast<int>(std::ceil(4 * sigma));
        cv::GaussianBlur(up, blurred, cv::Size(2 * radius + 1, 2 * radius + 1), sigma, sigma, cv::BORDER_REFLECT_101);
        blurred.convertTo(scaled, CV_8U, 255.0 / (1.5 * std::max(threshold, 1e-6f)));
        cv::applyColorMap(scaled, colour, cv::COLORMAP_JET);
        cv::cvtColor(cropped, base, cv::COLOR_RGB2BGR);
        cv::addWeighted(base, 0.55, colour, 0.45, 0.0, overlay);
        cv::imwrite(argv[5], overlay);
    }
    return 0;
}
'''

CPP_CMAKE = r'''cmake_minimum_required(VERSION 3.16)
project(patchcore_infer CXX)
set(CMAKE_CXX_STANDARD 17)
set(CMAKE_CXX_STANDARD_REQUIRED ON)
if(NOT CMAKE_BUILD_TYPE)
  set(CMAKE_BUILD_TYPE Release)
endif()
set(ONNXRUNTIME_DIR "" CACHE PATH "Folder of an unpacked onnxruntime release (has include/ and lib/)")
find_package(OpenCV QUIET COMPONENTS core imgproc imgcodecs)
if(NOT OpenCV_FOUND)
  find_package(PkgConfig REQUIRED)
  pkg_check_modules(OpenCV REQUIRED opencv4)
  set(OpenCV_LIBS ${OpenCV_LIBRARIES})
endif()
add_executable(patchcore_infer main.cpp)
target_include_directories(patchcore_infer PRIVATE ${ONNXRUNTIME_DIR}/include ${OpenCV_INCLUDE_DIRS})
target_link_libraries(patchcore_infer PRIVATE ${OpenCV_LIBS} ${ONNXRUNTIME_DIR}/lib/libonnxruntime.so)
set_target_properties(patchcore_infer PROPERTIES BUILD_RPATH ${ONNXRUNTIME_DIR}/lib)
target_compile_options(patchcore_infer PRIVATE -O3 -march=native)
'''

CPP_README = r'''Build (Linux):
  sudo apt-get install -y libopencv-dev cmake g++
  download and unpack onnxruntime-linux-x64-<version>.tgz from github.com/microsoft/onnxruntime/releases
  cmake -S . -B build -DONNXRUNTIME_DIR=/path/to/onnxruntime-linux-x64-<version>
  cmake --build build -j
Without CMake (tested with OpenCV 4.6 and ONNX Runtime 1.20.1):
  g++ -O3 -std=c++17 main.cpp -I$ORT/include -I/usr/include/opencv4 -L$ORT/lib -lonnxruntime \
      -lopencv_imgcodecs -lopencv_imgproc -lopencv_core -Wl,-rpath,$ORT/lib -o patchcore_infer
Run:
  ./build/patchcore_infer patchcore_wrn.onnx bank.bin meta.txt image.png overlay.png
Check:
  the printed score must match references.<tag>.python_score in parity_report.json to about 1e-3 relative.
'''


def write_cpp_sources(folder):
    cpp = Path(folder) / "cpp"
    cpp.mkdir(parents=True, exist_ok=True)
    (cpp / "main.cpp").write_text(CPP_MAIN)
    (cpp / "CMakeLists.txt").write_text(CPP_CMAKE)
    (cpp / "README.txt").write_text(CPP_README)


ORT_VERSION = "1.20.1"


def verify_cpp(folder, references, threshold):
    import subprocess, urllib.request
    ort_dir = Path("/tmp") / f"onnxruntime-linux-x64-{ORT_VERSION}"
    binary = Path(folder) / "cpp" / "patchcore_infer"
    try:
        if not ort_dir.exists():
            url = (f"https://github.com/microsoft/onnxruntime/releases/download/v{ORT_VERSION}/"
                   f"onnxruntime-linux-x64-{ORT_VERSION}.tgz")
            urllib.request.urlretrieve(url, "/tmp/onnxruntime.tgz")
            safe_extract("/tmp/onnxruntime.tgz", "/tmp")
        if not Path("/usr/include/opencv4").exists():
            subprocess.run(["apt-get", "update", "-qq"], check=True, capture_output=True)
            subprocess.run(["apt-get", "install", "-y", "-qq", "libopencv-dev"], check=True, capture_output=True)
        subprocess.run(["g++", "-O2", "-std=c++17", str(Path(folder) / "cpp" / "main.cpp"), f"-I{ort_dir}/include",
                        "-I/usr/include/opencv4", f"-L{ort_dir}/lib", "-lonnxruntime", "-lopencv_imgcodecs",
                        "-lopencv_imgproc", "-lopencv_core", f"-Wl,-rpath,{ort_dir}/lib", "-o", str(binary)],
                       check=True, capture_output=True, text=True)
    except Exception as error:
        detail = getattr(error, "stderr", "") or str(error)
        return {"verified": False, "reason": f"build failed: {str(detail).strip()[-300:]}"}
    out = {"verified": True, "references": {}}
    for tag, ref in references.items():
        run = subprocess.run([str(binary), str(Path(folder) / "patchcore_wrn.onnx"), str(Path(folder) / "bank.bin"),
                              str(Path(folder) / "meta.txt"), ref["image"]], capture_output=True, text=True)
        lines = dict(line.split(maxsplit=1) for line in run.stdout.splitlines() if " " in line)
        if run.returncode != 0 or "score" not in lines:
            out["verified"] = False
            out["references"][tag] = {"error": run.stderr.strip()[-200:]}
            continue
        cpp_score = float(lines["score"])
        gap = abs(cpp_score - ref["onnx_score"])
        scale = max(abs(ref["onnx_score"]), 1.0)
        same_decision = (cpp_score > threshold) == (ref["python_score"] > threshold)
        out["references"][tag] = {"cpp_score": cpp_score, "onnx_numpy_score": ref["onnx_score"],
                                  "python_score": ref["python_score"], "cpp_vs_onnx_numpy": gap,
                                  "same_decision_as_python": bool(same_decision)}
        out["verified"] &= bool(gap / scale < 1e-4 and same_decision)
    return out

In [11]:
def _aupro_exhaustive(maps, masks, valid=None, fpr_limit=0.3):
    valid = np.ones(masks.shape, bool) if valid is None else valid
    background = maps[(masks == 0) & valid]
    labelled = [ndimage.label((m > 0) & v, structure=np.ones((3, 3), int)) for m, v in zip(masks, valid)]
    regions = [maps[k][lab == r] for k, (lab, count) in enumerate(labelled) for r in range(1, count + 1)]
    points = []
    for t in np.unique(maps[valid])[::-1]:
        fpr = (background > t).mean()
        if fpr > fpr_limit:
            break
        points.append((fpr, np.mean([(r > t).mean() for r in regions])))
    fpr, pro = np.array(points).T
    grid = np.concatenate([[0.0], fpr[fpr < fpr_limit], [fpr_limit]])
    integrate = getattr(np, "trapezoid", None) or np.trapz
    return float(integrate(np.interp(grid, fpr, pro), grid) / fpr_limit)


def run_sanity_checks():
    from sklearn.metrics import roc_auc_score
    rng = np.random.default_rng(0)

    scores = np.round(rng.normal(size=600), 1)
    labels = rng.integers(0, 2, 600)
    scores[labels == 1] += 0.8
    auroc_gap = abs(auroc(scores[labels == 0], scores[labels == 1]) - roc_auc_score(labels, scores))
    assert auroc_gap < 1e-12, auroc_gap

    masks = np.zeros((12, 40, 40), np.uint8)
    for k in range(6):
        masks[k, 5 + 2 * k:15 + 2 * k, 8:20] = 1
    maps = rng.random(masks.shape) + 0.6 * masks
    aupro_gap = abs(aupro(maps, masks) - _aupro_exhaustive(maps, masks))
    assert aupro_gap < 0.01, aupro_gap

    valid = np.ones(masks.shape, bool)
    valid[:, :, :12] = False
    valid[::2, 30:, :] = False
    masked_gap = abs(aupro(maps, masks, valid) - _aupro_exhaustive(maps, masks, valid))
    assert masked_gap < 0.01, masked_gap
    pixel_gap = abs(pixel_auroc(maps, masks, valid) - roc_auc_score(masks[valid], maps[valid]))
    assert pixel_gap < 1e-12, pixel_gap
    noisy_maps, noisy_masks = maps.copy(), masks.copy()
    noisy_maps[~valid] = 100.0
    noisy_masks[~valid] = rng.integers(0, 2, (~valid).sum())
    assert aupro(noisy_maps, noisy_masks, valid) == aupro(maps, masks, valid)
    assert pixel_auroc(noisy_maps, noisy_masks, valid) == pixel_auroc(maps, masks, valid)

    n_cal, alpha = 40, CONFIG["alpha"]
    alarms = [rng.normal() > conformal_threshold(rng.normal(size=n_cal), alpha) for _ in range(4000)]
    expected = 1 - math.ceil((n_cal + 1) * (1 - alpha)) / (n_cal + 1)
    assert abs(np.mean(alarms) - expected) < 0.02, (np.mean(alarms), expected)

    size = CONFIG["image_size"]
    image = np.zeros((size, size, 3), np.uint8)
    mask = np.zeros((size, size), np.uint8)
    cv2.rectangle(image, (90, 100), (150, 170), (255, 255, 255), -1)
    cv2.rectangle(mask, (90, 100), (150, 170), 1, -1)
    worst_iou = 1.0
    for family in ("rotation", "perspective"):
        warped, warped_mask = apply_nuisance(image, mask, family, max(CONFIG["severities"]), ("sanity", family))
        bright = warped[..., 0] > 127
        worst_iou = min(worst_iou, (bright & (warped_mask > 0)).sum() / (bright | (warped_mask > 0)).sum())
    assert worst_iou > 0.95, worst_iou

    print(f"Sanity checks passed. AUROC matches scikit-learn, AUPRO is within {aupro_gap:.1e} of the exhaustive "
          f"version ({masked_gap:.1e} with a validity mask, and pixels outside it have no effect), the conformal "
          f"false-alarm rate is {np.mean(alarms):.3f} (expected {expected:.3f}), and warped masks overlap their "
          f"objects with IoU {worst_iou:.3f}.")

In [12]:
def restore_previous_output():
    import shutil
    out = Path(CONFIG["output_dir"])
    if (out / "categories").exists() and any((out / "categories").iterdir()):
        return
    name = out.name
    for base in CONFIG["previous_output_roots"]:
        if not os.path.isdir(base):
            continue
        for dirpath, dirnames, _ in os.walk(base, followlinks=True):
            if os.path.basename(dirpath) == name and os.path.isdir(os.path.join(dirpath, "categories")):
                for sub in ("categories", "deploy"):
                    if os.path.isdir(os.path.join(dirpath, sub)):
                        shutil.copytree(os.path.join(dirpath, sub), out / sub, dirs_exist_ok=True)
                log(f"Reusing finished categories from {dirpath}")
                return
            depth = dirpath[len(base):].count(os.sep)
            dirnames[:] = [] if depth >= 6 else [x for x in dirnames if x not in ("categories", "figures", "deploy")]


def run_all():
    out = Path(CONFIG["output_dir"])
    out.mkdir(parents=True, exist_ok=True)
    run_sanity_checks()
    restore_previous_output()
    set_seed(CONFIG["seed"])
    with open(out / "config.json", "w") as handle:
        json.dump(CONFIG, handle, indent=1, default=str)
    models = load_models()
    categories = available_categories()
    log(f"Categories: {', '.join(categories)}")
    metrics = pd.concat([run_category(c, models) for c in categories], ignore_index=True)
    metrics.to_csv(out / "metrics_all.csv", index=False)

    records, thresholds = load_records(categories)
    summary = summary_table(metrics)
    transfer = augmentation_transfer_table(metrics)
    exchange_detail, exchange = exchangeability_table(records)
    testcal_detail, testcal_images = test_split_calibration(records)
    alarms = rate_table(pd.concat([train_image_level(records, thresholds), testcal_images], ignore_index=True))
    hypotheses = test_hypotheses(metrics, records, testcal_detail)
    ndr_frame, ndr_summary = ndr_analysis(metrics)
    families = per_family_table(metrics)
    margins = margin_sensitivity(metrics)
    margins.to_csv(out / "margin_sensitivity.csv")
    tradeoff = calibration_tradeoff_table(alarms)
    controls = {"C1 colour-specific cost (qualifies H3)": colour_specific_control(records, "patchcore_wrn", "patchcore_wrn_aug"),
                "C2 reflected-border control for geometric shifts (qualifies H2 and H4)": geometric_border_control(metrics),
                "C3 NDR at fixed severity (qualifies the NDR result)": ndr_fixed_severity(ndr_frame)}
    with open(out / "controls.json", "w") as handle:
        json.dump(controls, handle, indent=1, default=float)
    families.to_csv(out / "per_family.csv")
    followup, followup_tradeoff, followup_descriptive = followup_tests(metrics, records)
    followup.to_csv(out / "followup_local_contrast.csv")
    explore = exploratory_local(metrics, records)
    for name, frame in [("summary", summary), ("augmentation_transfer", transfer), ("false_alarms", alarms),
                        ("hypotheses", hypotheses), ("exchangeability", exchange_detail), ("test_split_calibration", testcal_detail)]:
        frame.to_csv(out / f"{name}.csv")

    figure_robustness_curves(metrics)
    figure_confusion(metrics)
    figure_invariance_sensitivity(metrics)
    figure_false_alarms(alarms)
    figure_ndr(ndr_frame)
    figure_local_contrast(metrics)
    for c in categories:
        figure_gallery(c)

    deployment = {}
    try:
        deployment = export_deployment(models)
    except Exception as error:
        log(f"Deployment export failed: {error}")

    few = len(categories) < 6
    banner = {"Run": f"Only {len(categories)} categories were found, so the p-values below are not meaningful." if few
              else f"Full run on {len(categories)} categories."}
    text = write_report({**banner,
                         "Detector x regime (mean of per-category means)": summary,
                         "Augmentation gain by nuisance group, seen vs held-out": transfer,
                         "Calibration exchangeability: train-split calibration normals vs clean test normals": exchange,
                         "False alarms by calibration policy (cluster bootstrap CI, method in ci_method)": alarms,
                         "Pre-registered hypotheses (H3 and E2 use a sign-flip test over categories; p_image_level is the original image-level p)":
                             hypotheses[["id", "statement", "unit", "n", "mean_difference", "ci_low", "ci_high", "p", "holm_p",
                                         "min_attainable_p", "images", "p_image_level", "verdict"]],
                         "Calibration trade-off under held-out shift (test-split calibration)": tradeoff,
                         "Image AUROC by family (raw and real-content-only scores)": families,
                         "Nuisance displacement ratio vs AUROC drop": ndr_summary,
                         "Border margin sensitivity (geometric results scored with larger real-content margins)": margins,
                         "Post hoc controls (added after the first full run; they qualify, never replace, the hypotheses)": controls,
                         "Follow-up pre-registration": FOLLOWUP_PREREGISTRATION,
                         "Follow-up tests (local-contrast score)": followup[["id", "statement", "n", "mean_difference", "ci_low", "ci_high", "p", "holm_p", "verdict"]],
                         "Follow-up calibration trade-off (PatchCore WRN, held-out, test-split)": followup_tradeoff,
                         "Max score vs local-contrast score, all detectors": followup_descriptive,
                         "Exploratory, after the follow-up run: lowest held-out false-alarm rate reachable with clean calibration":
                             explore["lowest_heldout_false_alarm_reachable_with_clean_calibration"],
                         "Exploratory: textures vs objects": explore["texture_vs_object"],
                         "Exploratory: per-category held-out false alarms at the clean 90th percentile (PatchCore WRN)":
                             explore["per_category_heldout_patchcore_wrn"],
                         "Deployment (ONNX parity, CPU timing, C++ check)": deployment})
    print(text)
    return dict(metrics=metrics, records=records, summary=summary, transfer=transfer, alarms=alarms, exchangeability=exchange,
                test_split=testcal_detail, hypotheses=hypotheses, followup=followup, ndr=ndr_summary, deployment=deployment)


results = run_all()

Sanity checks passed. AUROC matches scikit-learn, AUPRO is within 2.3e-05 of the exhaustive version (2.4e-05 with a validity mask, and pixels outside it have no effect), the conformal false-alarm rate is 0.096 (expected 0.098), and warped masks overlap their objects with IoU 0.999.
Downloading: "https://download.pytorch.org/models/wide_resnet50_2-95faca4d.pth" to /root/.cache/torch/hub/checkpoints/wide_resnet50_2-95faca4d.pth


100%|██████████| 132M/132M [00:00<00:00, 187MB/s]


model.safetensors: reconstructing file:   0%|          |  0.00B / 88.2MB            

model.safetensors: downloading bytes:           |  0.00B            

Extracting mvtec_anomaly_detection.tar.xz to /tmp/mvtec_extracted. This takes a few minutes the first time.
MVTec AD found at /tmp/mvtec_extracted
[20:12:01] Categories: bottle, cable, capsule, carpet, grid, hazelnut, leather, metal_nut, pill, screw, tile, toothbrush, transistor, wood, zipper
[20:12:08] bottle: fit 167, cal 42, test 83 (63 anomalous), 22 conditions, 7 detectors
[20:12:52] bottle: detectors fitted in 51s
[20:15:24] bottle: done in 203s
[20:15:38] cable: fit 179, cal 45, test 150 (92 anomalous), 22 conditions, 7 detectors
[20:16:28] cable: detectors fitted in 64s
[20:20:52] cable: done in 328s
[20:21:06] capsule: fit 175, cal 44, test 132 (109 anomalous), 22 conditions, 7 detectors
[20:21:54] capsule: detectors fitted in 62s
[20:25:42] capsule: done in 290s
[20:25:57] carpet: fit 224, cal 56, test 117 (89 anomalous), 22 conditions, 7 detectors
[20:27:05] carpet: detectors fitted in 83s
[20:30:48] carpet: done in 306s
[20:30:54] grid: fit 211, cal 53, test 78 (57 anomalou